In [ ]:
import torch
print(torch.__version__)  # should print cleanly

In [ ]:
!pip install numpy pandas networkx matplotlib tqdm -q


# **DATASET CREATION**




In [ ]:
import numpy as np
import pandas as pd
import random
from tqdm import tqdm




In [ ]:
class TrackSegment:
    def __init__(self, segment_id, length=1000):
        self.id     = segment_id
        self.length = length

        # Use segment_id as seed so each segment gets different but
        # reproducible properties — fixes zero-variance bug
        rng = np.random.RandomState(seed=segment_id * 17 + 3)

        self.gradient    = rng.uniform(-0.02,  0.02)   # slope: -2% to +2%
        self.curvature   = rng.uniform( 0.001, 0.012)  # gentle to sharp curve
        self.speed_limit = rng.uniform(20.0,   80.0)   # m/s: slow urban to fast mainline
        self.friction    = rng.uniform( 0.70,  0.95)   # dry=0.95, wet=0.75

        # Signal starts green; updated by occupancy logic, not random noise
        self.signal_state = 0   # 0=Green, 1=Yellow, 2=Red

    def update_signal(self, trains_on_segment, all_trains):
        """
        Red   : another train is within 50m on same segment
        Yellow: another train is within 150m on same segment
        Green : segment clear
        """
        min_gap = float("inf")
        for t1 in trains_on_segment:
            for t2 in all_trains:
                if t2.id == t1.id:
                    continue
                gap = t2.position - t1.position
                if 0 < gap < min_gap:
                    min_gap = gap

        if   min_gap < 50:
            self.signal_state = 2   # Red
        elif min_gap < 150:
            self.signal_state = 1   # Yellow
        else:
            self.signal_state = 0   # Green

In [ ]:
from collections import deque

class Train:
    def __init__(self, train_id):
        self.id           = train_id
        self.position     = np.random.uniform(0, 200)
        self.velocity     = np.random.uniform(10, 20)   # m/s
        self.acceleration = 0.0
        self.max_accel    = 1.2
        self.max_brake    = 2.5
        self.driver_type  = random.choice(["aggressive", "normal", "lazy"])
        self.position_history = deque(maxlen=5)

    def update(self, dt, segment):
        if   self.driver_type == "aggressive":
            accel = np.random.uniform(0.5, self.max_accel)
        elif self.driver_type == "lazy":
            accel = np.random.uniform(0.1, 0.6)
        else:
            accel = np.random.uniform(0.3, 0.9)

        # Emergency brake event
        if np.random.rand() < 0.002:
            accel = -self.max_brake

        # Signal-based braking — yellow = caution, red = brake hard
        if segment.signal_state == 2 and self.velocity > 5:
            accel = -self.max_brake * 0.8
        elif segment.signal_state == 1 and self.velocity > 8:
            accel = -self.max_brake * 0.3

        # Gradient effect on acceleration
        accel -= segment.gradient * 9.81

        self.acceleration = accel
        self.velocity     = float(np.clip(self.velocity + accel * dt,
                                          0, segment.speed_limit))
        self.position    += self.velocity * dt
        self.position_history.append(self.position)



In [ ]:
np.random.seed(42)
random.seed(42)

num_segments = 8
track_segments = [TrackSegment(i) for i in range(num_segments)]

num_trains = 8
timesteps  = 3000
dt         = 1

trains  = [Train(f"T{i}") for i in range(num_trains)]
records = []

print("Segment properties (should all be different now):")
for seg in track_segments:
    print(f"  Seg {seg.id}: gradient={seg.gradient:+.4f}  "
          f"curvature={seg.curvature:.4f}  "
          f"speed_limit={seg.speed_limit:.1f}  "
          f"friction={seg.friction:.3f}")

print(f"\nGenerating {num_trains} trains × {timesteps} timesteps = "
      f"{num_trains*timesteps:,} rows...")

for t in tqdm(range(timesteps)):

    # ── Update signals based on actual train proximity ─────────
    for seg in track_segments:
        trains_on_seg = [
            train for train in trains
            if int(train.position // 1000) % num_segments == seg.id
        ]
        seg.update_signal(trains_on_seg, trains)

    # ── Update train physics ────────────────────────────────────
    for train in trains:
        seg_idx = int(train.position // 1000) % num_segments
        segment = track_segments[seg_idx]
        train.update(dt, segment)

    # ── Collect data ────────────────────────────────────────────
    for train in trains:
        seg_idx = int(train.position // 1000) % num_segments
        segment = track_segments[seg_idx]

        # Communication delay (0–2 steps)
        delay_steps = np.random.randint(0, 3)
        hist = list(train.position_history)
        delayed_position = (hist[-delay_steps - 1]
                            if len(hist) > delay_steps
                            else train.position)

        # Sensor noise — position σ=3m, velocity σ=0.5 m/s
        measured_position = delayed_position + np.random.normal(0, 3)
        measured_velocity = train.velocity    + np.random.normal(0, 0.5)

        # ── True headway (ground truth, pre-noise) ──────────────────
        # Used for LABEL only — labels must be ground truth, not sensor readings
        min_headway      = 9999.0
        closing_velocity = 0.0
        for other in trains:
            if other.id == train.id:
                continue
            dist = other.position - train.position
            if 0 < dist < min_headway:
                min_headway      = dist
                closing_velocity = train.velocity - other.velocity

        # ── Noisy estimated headway (sensor reading) → features only ─
        estimated_headway = float(np.clip(
            min_headway + np.random.normal(0, 2), 0, None
        ))

        # ── Ground truth label ────────────────────────────────────────
        # Theoretical stopping distance — assumes max braking from NOW
        # Independent of what driver is currently doing
        # Answers: "can this train physically stop before hitting the one ahead?"
        stopping_distance = (
            (train.velocity ** 2) /
            (2 * train.max_brake * max(segment.friction, 0.01))
        )

        # True TTC at current relative velocity — no braking assumed
        if closing_velocity > 0.5:
            true_ttc = min_headway / closing_velocity
        else:
            true_ttc = 9999.0

        # Label = 1 if collision unavoidable without maximum intervention
        # Uses min_headway (truth) not estimated_headway (noisy)
        # Uses stopping_distance (theoretical) not current braking (driver response)
        collision_risk = int(
    (min_headway < stopping_distance   # physically can't stop
     and true_ttc < 20                 # tighter: impact within 20s
     and train.velocity > 5            # must be moving meaningfully
     and closing_velocity > 1.0)       # must be genuinely closing
    or (segment.signal_state == 2 and train.velocity > 10)
)

        records.append({
            "timestamp":        t,
            "train_id":         train.id,
            "position":         measured_position,
            "velocity":         measured_velocity,
            "acceleration":     train.acceleration,
            "estimated_headway": estimated_headway,
            "segment_id":       segment.id,
            "gradient":         segment.gradient,
            "curvature":        segment.curvature,
            "speed_limit":      segment.speed_limit,
            "signal_state":     segment.signal_state,
            "friction":         segment.friction,
            "collision_risk":   collision_risk,
        })

df = pd.DataFrame(records)
print(f"\nDataset shape: {df.shape}")


In [ ]:
print("=" * 55)
print("DATASET VALIDATION")
print("=" * 55)

# 1. Segment variance
for col in ["gradient", "curvature", "speed_limit", "friction", "segment_id"]:
    n_unique = df[col].nunique()
    variance = df[col].var()
    status   = "✅" if n_unique > 1 else "❌ STILL CONSTANT"
    print(f"{status}  {col:<20} unique={n_unique:>3}  var={variance:.6f}")

# 2. Signal state distribution
sig_counts = df["signal_state"].value_counts().sort_index()
print(f"\n✅  signal_state distribution:")
labels = {0: "Green", 1: "Yellow", 2: "Red"}
for state, count in sig_counts.items():
    pct = count / len(df) * 100
    flag = "⚠️ " if (state == 2 and pct < 1) else "  "
    print(f"    {flag}{labels[state]:>8} ({state}): {count:>6} rows  ({pct:.1f}%)")

# 3. Collision risk balance
risk_counts = df["collision_risk"].value_counts().sort_index()
pos_rate    = risk_counts.get(1, 0) / len(df) * 100
flag = "⚠️ " if pos_rate > 20 else "✅ "
print(f"\n{flag} collision_risk=1: {risk_counts.get(1,0):>6} rows  ({pos_rate:.1f}%)")
print(f"   collision_risk=0: {risk_counts.get(0,0):>6} rows  ({100-pos_rate:.1f}%)")
if pos_rate > 20:
    print("   ⚠️  Positive rate still high — check braking_distance threshold")
else:
    print("   ✅  Positive rate realistic (real ops: <10%)")

# 4. Headway sanity
neg_hw = (df["estimated_headway"] < 0).sum()
print(f"\n{'✅' if neg_hw == 0 else '❌'}  Negative headway rows: {neg_hw}")

# 5. Quick feature variance table
print(f"\nFeature variance summary:")
for col in ["velocity", "acceleration", "estimated_headway",
            "gradient", "curvature", "speed_limit",
            "signal_state", "friction"]:
    print(f"  {col:<25} var={df[col].var():.6f}  "
          f"min={df[col].min():.3f}  max={df[col].max():.3f}")

print(f"\nLabel sanity — mean feature values by class:")
print(f"  {'Feature':<25} {'risk=1':>10}  {'risk=0':>10}  {'direction':>10}  status")
print("  " + "-"*65)
risky = df[df["collision_risk"] == 1]
safe  = df[df["collision_risk"] == 0]
expected_direction = {
    "estimated_headway": "lower",
    "velocity":          "higher",
    "acceleration":      "either",   # should NOT dominate
}
for col, expected in expected_direction.items():
    r = risky[col].mean()
    s = safe[col].mean()
    actual = "lower" if r < s else "higher"
    if expected == "either":
        flag = "✅" if abs(r - s) < 0.5 else "⚠️  dominated by accel — label bug"
    else:
        flag = "✅" if actual == expected else "❌ WRONG DIRECTION"
    print(f"  {col:<25} {r:>10.3f}  {s:>10.3f}  {actual:>10}  {flag}")

print("\n" + "=" * 55)
print("df.head()")
print("=" * 55)
df.head(16)

In [ ]:
df.to_csv("train_simulation_dataset.csv", index=False)


# ***ML PREPARATION***


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

features = [
    "velocity",
    "acceleration",
    "estimated_headway",
    "gradient",
    "curvature",
    "speed_limit",
    "signal_state",
    "friction",
]
# 8 features — position and segment_id dropped
df["estimated_headway"] = df["estimated_headway"].clip(upper=500)
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

# Save scaler — needed to transform live data in visualisation
import pickle
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved to scaler.pkl")
print(f"   Fitted on {len(features)} features: {features}")

In [ ]:
sequence_length = 10

X = []
y = []

for train_id in df["train_id"].unique():

    train_df = df[df["train_id"] == train_id]

    values = train_df[features].values
    targets = train_df["collision_risk"].values

    for i in range(len(train_df) - sequence_length):
        X.append(values[i:i+sequence_length])
        y.append(targets[i+sequence_length])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
import torch
import torch.nn as nn

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1).to(device)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

In [ ]:
class CollisionLSTM(nn.Module):
    def __init__(self, input_size=8, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc1  = nn.Linear(hidden_size, 32)
        self.fc2  = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = self.relu(self.fc1(out[:, -1, :]))
        return self.fc2(out)   # raw logits — BCEWithLogitsLoss handles sigmoid

# pos_weight handles class imbalance — keep from original
positive_weight = torch.tensor([10.0], device=device)  # cap at 10, don't let it go to 51
print(f"pos_weight: {positive_weight.item():.2f}  "
      f"(higher = rarer positive class)")

criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
model     = CollisionLSTM(input_size=len(features)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: CollisionLSTM | params={total_params:,} | input_size={len(features)}")


In [ ]:
epochs = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss    = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss    = criterion(test_outputs, y_test)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {loss.item():.4f} | "
          f"Test Loss: {test_loss.item():.4f}")

In [ ]:
# Add this after Cell 10 as a new cell
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_test)).cpu().numpy().flatten()

y_true = y_test.cpu().numpy().flatten()

# Find threshold that maximises F1
precisions, recalls, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx   = f1_scores.argmax()
best_thresh = thresholds[best_idx]

print(f"Best threshold: {best_thresh:.3f}")
print(f"At best threshold — Precision: {precisions[best_idx]:.4f}  "
      f"Recall: {recalls[best_idx]:.4f}  F1: {f1_scores[best_idx]:.4f}")

# Compare default vs optimal
preds_default = (probs > 0.5).astype(float)
preds_optimal = (probs > best_thresh).astype(float)
print(f"\nDefault  (0.50): F1={f1_score(y_true, preds_default):.4f}  "
      f"Precision={preds_default[y_true==1].mean():.4f}")
print(f"Optimal  ({best_thresh:.2f}): F1={f1_score(y_true, preds_optimal):.4f}")

In [ ]:
# In Cell 7, after building X and y — add this
from collections import Counter

print(f"Before balancing: {Counter(y)}")

# Find indices of positive samples
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

# Upsample positives to 10% of dataset
target_pos = int(len(neg_idx) * 0.10)  # 10% positive rate
upsample_idx = np.random.choice(pos_idx, size=target_pos, replace=True)
balanced_idx = np.concatenate([neg_idx, upsample_idx])
np.random.shuffle(balanced_idx)

X = X[balanced_idx]
y = y[balanced_idx]

print(f"After balancing:  {Counter(y)}")
print(f"New positive rate: {y.mean()*100:.1f}%")

In [ ]:
!pip install umap-learn

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

preds = (probs > best_thresh).astype(float)
cm = confusion_matrix(y_true, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Safe", "Risky"])
disp.plot(cmap="Blues")
plt.title("Collision Risk Prediction — Confusion Matrix")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
from sklearn.metrics import PrecisionRecallDisplay
PrecisionRecallDisplay.from_predictions(y_true, probs)
plt.title("Precision-Recall Curve")
plt.savefig("pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()
train_losses, test_losses = [], []

plt.figure(figsize=(8,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses,  label="Test Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("LSTM Training Convergence")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
import umap
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    # Get hidden states — shape (n_samples, 64)
    _, (h_n, _) = model.lstm(X_test)
    hidden = h_n[-1].cpu().numpy()   # last layer, shape (n_test, 64)

reducer   = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(hidden)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(embedding[:,0], embedding[:,1],
                      c=y_test.cpu().numpy().flatten(),
                      cmap="RdYlGn_r", alpha=0.5, s=8)
plt.colorbar(scatter, label="Collision Risk")
plt.title("UMAP of LSTM Hidden States\n(green=safe, red=risky)")
plt.savefig("umap_hidden.png", dpi=150, bbox_inches="tight")
plt.show()

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Flatten sequences for non-sequential models
X_tr_flat = X_train.cpu().numpy().reshape(len(X_train), -1)
X_te_flat = X_test.cpu().numpy().reshape(len(X_test), -1)
y_tr_flat = y_train.cpu().numpy().flatten()
y_te_flat = y_test.cpu().numpy().flatten()

for name, clf in [
    ("Logistic Regression", LogisticRegression(max_iter=500, class_weight="balanced")),
    ("Random Forest",       RandomForestClassifier(n_estimators=100, class_weight="balanced"))
]:
    clf.fit(X_tr_flat, y_tr_flat)
    preds = clf.predict(X_te_flat)
    print(f"\n{name}")
    print(classification_report(y_te_flat, preds, target_names=["Safe","Risky"]))



# **Track Occupancy Prediction and Route Optimsation**

In [ ]:
"""
Enhanced Railway Train Monitoring System - Google Colab Version
─────────────────────────────────────────────────────────────────
FEATURES:
  ✅ Track Occupancy Heatmap  — real segment load, rolls over time
  ✅ Auto Route Optimisation  — Dijkstra + congestion cost penalty
  ✅ Major Stations (MUST visit) vs Minor Stations (can bypass)
  ✅ Live reroute counter + cyan flash ring on map
─────────────────────────────────────────────────────────────────
"""

# !pip install networkx torch -q
# %matplotlib inline

from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import random
from collections import deque
import heapq
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.ERROR)
logging.getLogger('torch').setLevel(logging.ERROR)

# ══════════════════════════════════════════════════════════════
# 1. NETWORK
# ══════════════════════════════════════════════════════════════
G = nx.Graph()

pos = {
    "A": (0,  5),   # MAJOR – Western Terminus
    "C": (6,  5),   # MAJOR – Central Junction
    "E": (12, 5),   # MAJOR – Eastern Terminus
    "G": (9,  8),   # MAJOR – Northern Hub
    "H": (3,  2),   # MAJOR – Southern Gate
    "B": (3,  5),   # minor
    "D": (9,  5),   # minor
    "F": (6,  8),   # minor
    "I": (6,  2),   # minor
    "J": (9,  2),   # minor
    "K": (12, 8),   # minor
}

MAJOR_STATIONS = {"A","C","E","G","H"}
MINOR_STATIONS = {"B","D","F","I","J","K"}

weighted_edges = [
    ("A","B",3),("B","C",3),("C","D",3),("D","E",3),
    ("C","F",3),("F","G",3),("G","K",3),("K","E",3),
    ("B","H",3),("H","I",3),("I","C",3),
    ("D","J",3),("J","E",3),
    ("G","D",3),
]
G.add_weighted_edges_from(weighted_edges)

# ── Canonical edge lookup (KEY FIX) ──────────────────────────
# G.edges() returns both (u,v) and (v,u) as valid for undirected graphs,
# so `(a,b) in G.edges()` is always True — we can't rely on that.
# Instead, build an explicit dict: both orderings → one canonical tuple.
_CANONICAL = {}
for u, v in G.edges():
    _CANONICAL[(u, v)] = (u, v)
    _CANONICAL[(v, u)] = (u, v)   # reverse maps to same canonical

def canonical_edge(a, b):
    return _CANONICAL[(a, b)]     # raises KeyError if truly not an edge

# Segment list: one entry per physical track segment
SEGMENTS = list({_CANONICAL[(u,v)] for u,v in G.edges()})
SEGMENTS.sort()   # stable order for heatmap rows

# ══════════════════════════════════════════════════════════════
# 2. OCCUPANCY TRACKING  (deterministic, no untrained NN)
# ══════════════════════════════════════════════════════════════
HIST_LEN       = 40
MAX_ON_SEG     = 3          # normalisation cap
REROUTE_THRESH = 0.30       # congestion above this triggers rerouting

seg_history = {s: deque([0]*HIST_LEN, maxlen=HIST_LEN) for s in SEGMENTS}
seg_load    = {s: 0   for s in SEGMENTS}
seg_pred    = {s: 0.0 for s in SEGMENTS}

def update_occupancy(trains):
    """Count trains per segment, compute smoothed congestion score 0-1."""
    for s in SEGMENTS:
        seg_load[s] = 0
    for train in trains:
        e = train.current_edge()
        if e:
            s = canonical_edge(*e)
            seg_load[s] += 1
    for s in SEGMENTS:
        seg_history[s].append(seg_load[s])
        recent      = list(seg_history[s])[-10:]
        avg         = np.mean(recent) / MAX_ON_SEG
        curr        = seg_load[s]    / MAX_ON_SEG
        seg_pred[s] = float(np.clip(0.6*avg + 0.4*curr, 0.0, 1.0))

# ══════════════════════════════════════════════════════════════
# 3. DIJKSTRA WITH CONGESTION COST
# ══════════════════════════════════════════════════════════════
CONGESTION_PENALTY = 15.0

def build_cost_graph():
    cost = {}
    for u, v, d in G.edges(data=True):
        base    = d.get("weight", 3)
        s       = canonical_edge(u, v)
        penalty = CONGESTION_PENALTY * seg_pred[s] if seg_pred[s] > REROUTE_THRESH else 0
        w = base + penalty
        cost.setdefault(u, {})[v] = w
        cost.setdefault(v, {})[u] = w
    return cost

def dijkstra(graph, src, dst):
    dist = {src: 0}
    prev = {}
    pq   = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, float("inf")):
            continue
        if u == dst:
            break
        for v, w in graph.get(u, {}).items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    path, cur = [], dst
    while cur != src:
        path.append(cur)
        cur = prev.get(cur)
        if cur is None:
            return None
    path.append(src)
    return list(reversed(path))

def mandatory_route(waypoints):
    """Chain Dijkstra hops between consecutive mandatory stops."""
    cg   = build_cost_graph()
    full = []
    for i in range(len(waypoints) - 1):
        seg = dijkstra(cg, waypoints[i], waypoints[i+1])
        if seg is None:
            return None
        full = full[:-1] + seg if full else seg
    return full

# ══════════════════════════════════════════════════════════════
# 4. TRAIN
# ══════════════════════════════════════════════════════════════
TRAIN_MANDATORY = {
    "T1": ["A","C","E"],        # west → east main corridor
    "T2": ["E","G","C","A"],    # east → west via north hub  (crosses T1 at C)
    "T3": ["H","C","G","E"],    # south gate → north hub → east
    "T4": ["K","G","C","H"],    # north-east → south gate  (crosses T2/T3 at G,C)
    "T5": ["J","E","C","A"],    # south-east → west  (crosses T1 at C, T2 at C)
}

class Train2D:
    def __init__(self, tid, speed_range=(0.012, 0.025)):
        self.train_id        = tid
        self.mandatory       = TRAIN_MANDATORY[tid]
        self.path            = mandatory_route(self.mandatory) or self.mandatory[:]
        self.edge_index      = 0
        self.progress        = random.uniform(0, 0.15)
        self.base_speed      = random.uniform(*speed_range)
        self.speed           = self.base_speed
        self.color           = "#4CAF50"
        self.history         = deque(maxlen=50)
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.velocity        = 0.0
        self.collision_risk  = 0.0
        self.reroute_count   = 0
        self.last_reroute    = -999
        self.reroute_flash   = 0

    def current_edge(self):
        if self.edge_index >= len(self.path) - 1:
            return None
        return (self.path[self.edge_index], self.path[self.edge_index + 1])

    def apply_brake(self, intensity):
        self.is_braking      = True
        self.brake_intensity = intensity
        self.speed           = self.base_speed * (1 - intensity * 0.8)

    def release_brake(self):
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.speed           = self.base_speed

    def update(self):
        self.progress += self.speed
        if self.progress >= 1:
            self.progress   = 0
            self.edge_index = (self.edge_index + 1) % max(1, len(self.path) - 1)
        if self.reroute_flash > 0:
            self.reroute_flash -= 1

    def get_position(self):
        if self.edge_index >= len(self.path) - 1:
            return pos[self.path[-1]]
        x1, y1 = pos[self.path[self.edge_index]]
        x2, y2 = pos[self.path[self.edge_index + 1]]
        return (x1 + (x2-x1)*self.progress, y1 + (y2-y1)*self.progress)

    def try_reroute(self, frame):
        if frame - self.last_reroute < 25:
            return False
        if self.edge_index >= len(self.path) - 1:
            return False
        current_node   = self.path[self.edge_index]
        visited_majors = {n for n in self.path[:self.edge_index+1] if n in MAJOR_STATIONS}
        remaining      = [m for m in self.mandatory if m not in visited_majors]
        if not remaining:
            return False
        new_suffix = mandatory_route([current_node] + remaining)
        if new_suffix is None:
            return False
        old_suffix = self.path[self.edge_index:]
        if new_suffix == old_suffix:
            return False
        self.path          = self.path[:self.edge_index] + new_suffix
        self.progress      = 0
        self.reroute_count += 1
        self.last_reroute  = frame
        self.reroute_flash = 18
        return True

# ══════════════════════════════════════════════════════════════
# 5. COLLISION RISK MODEL
# ══════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class RiskModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(12, 64, 2, batch_first=True, dropout=0.2)
        self.fc1  = nn.Linear(64, 32)
        self.fc2  = nn.Linear(32, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc2(self.relu(self.fc1(out[:, -1, :])))

risk_model = RiskModel().to(device)
risk_model.eval()

CRITICAL_DIST = 0.5
WARNING_DIST  = 1.0
SAFE_DIST     = 2.0
SEQ_LEN       = 10

def compute_risk(train, others):
    min_dist = 999.0
    ce = train.current_edge()
    if ce is None:
        return 0.0
    for other in others:
        if other.train_id == train.train_id:
            continue
        oe = other.current_edge()
        if oe and (oe == ce or oe == (ce[1], ce[0])):
            d = np.linalg.norm(np.array(train.get_position()) - np.array(other.get_position()))
            min_dist = min(min_dist, d)

    vel  = train.speed * 100
    acc  = (train.speed - train.base_speed) * 100
    bd   = (vel**2) / 3.0 if vel > 0 else 0
    occ  = seg_pred.get(canonical_edge(*ce), 0.0)

    feat = np.array([
        vel, acc, bd, min_dist,
        random.uniform(-0.02, 0.02),
        0.005 if train.edge_index % 3 == 0 else 0.001,
        80, random.uniform(0, 0.3),
        1 if min_dist < WARNING_DIST else 0,
        min_dist / vel if vel > 0 else 999,
        min_dist - bd,
        train.brake_intensity,
    ], dtype=np.float32)
    train.history.append(feat)

    if len(train.history) < SEQ_LEN:
        prob = occ * 0.4
    else:
        seq = np.array(list(train.history)[-SEQ_LEN:])
        t   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            prob = torch.sigmoid(risk_model(t)).item()
        prob = 0.5*prob + 0.5*occ   # blend with real congestion

    train.velocity       = vel
    train.collision_risk = prob

    if   min_dist < CRITICAL_DIST: train.apply_brake(min(1.0, (CRITICAL_DIST-min_dist)/CRITICAL_DIST))
    elif min_dist < WARNING_DIST:  train.apply_brake(0.5)
    elif min_dist < SAFE_DIST:     train.apply_brake(0.2)
    else:                          train.release_brake()

    return prob

# ══════════════════════════════════════════════════════════════
# 6. CREATE TRAINS
# ══════════════════════════════════════════════════════════════
speed_ranges = {
    "T1":(0.015,0.025),
    "T2":(0.011,0.019),
    "T3":(0.013,0.022),
    "T4":(0.012,0.020),
    "T5":(0.014,0.023),
}
trains = [Train2D(tid, speed_ranges[tid]) for tid in TRAIN_MANDATORY]

print(f"{len(trains)} trains | {len(G.nodes())} stations "
      f"({len(MAJOR_STATIONS)} major / {len(MINOR_STATIONS)} minor) | "
      f"{len(G.edges())} segments\n")
for t in trains:
    print(f"  {t.train_id}: mandatory={t.mandatory}   path={t.path}")

# ══════════════════════════════════════════════════════════════
# 7. DASHBOARD
# ══════════════════════════════════════════════════════════════
plt.style.use('dark_background')
fig = plt.figure(figsize=(24, 14))
gs  = fig.add_gridspec(3, 4, hspace=0.44, wspace=0.40)

ax_net     = fig.add_subplot(gs[:2, :2])
ax_risk    = fig.add_subplot(gs[0,  2])
ax_speed   = fig.add_subplot(gs[1,  2])
ax_occ     = fig.add_subplot(gs[0,  3])
ax_head    = fig.add_subplot(gs[2,  0])
ax_brake   = fig.add_subplot(gs[2,  1])
ax_reroute = fig.add_subplot(gs[2,  2])
ax_stats   = fig.add_subplot(gs[2,  3])

# ── Network base ──────────────────────────────────────────────
nx.draw_networkx_edges(G, pos=pos, ax=ax_net,
                       edge_color='#455A64', width=6, alpha=0.55)
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MAJOR_STATIONS),
                       node_color='#FFD700', node_size=1600,
                       node_shape='s', alpha=0.95)
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MINOR_STATIONS),
                       node_color='#78909C', node_size=800,
                       node_shape='o', alpha=0.85)
nx.draw_networkx_labels(G, pos=pos, ax=ax_net,
                        font_color='white', font_weight='bold', font_size=11)

load_cmap  = plt.cm.RdYlGn_r
edge_lines = {}
for (u, v) in SEGMENTS:
    x1,y1 = pos[u]; x2,y2 = pos[v]
    ln, = ax_net.plot([x1,x2],[y1,y2], lw=7, alpha=0.0, zorder=2,
                      solid_capstyle='round')
    edge_lines[(u,v)] = ln

major_p = mpatches.Patch(color='#FFD700', label='Major Station')
minor_p = mpatches.Patch(color='#78909C', label='Minor Station')
free_p  = mpatches.Patch(color='#4CAF50', label='Track Free')
busy_p  = mpatches.Patch(color='#FF5722', label='Track Congested')
ax_net.legend(handles=[major_p,minor_p,free_p,busy_p],
              loc='lower right', fontsize=8)

train_scat = ax_net.scatter([], [], s=460, zorder=6,
                             edgecolors='white', linewidths=2.5)
brake_circles   = [Circle((0,0), 0.40, fill=False, edgecolor='#FF1744',
                            linewidth=4, visible=False, zorder=7) for _ in trains]
reroute_circles = [Circle((0,0), 0.58, fill=False, edgecolor='#00E5FF',
                            linewidth=3, linestyle='--', visible=False, zorder=7)
                   for _ in trains]
for c in brake_circles + reroute_circles:
    ax_net.add_patch(c)

tlabels = [ax_net.text(0, 0, '', fontsize=8, color='white', ha='center',
                        va='bottom', fontweight='bold', zorder=8)
           for _ in trains]

ax_net.set_title("🚂  Railway Network — Live Positions + Track Load",
                  fontsize=14, fontweight='bold', pad=12)
ax_net.set_xlim(-1.2, 14); ax_net.set_ylim(0.2, 10.5); ax_net.axis('off')

# ── Risk / Speed ──────────────────────────────────────────────
colors_t    = ['#00E5FF', '#FF6D00', '#69FF47', '#FF4DFF', '#FFD700']  # cyan, orange, lime, magenta, gold
risk_lines  = [ax_risk.plot([],[],  color=c, lw=2, label=t.train_id)[0]
               for t, c in zip(trains, colors_t)]
speed_lines = [ax_speed.plot([],[], color=c, lw=2, label=t.train_id)[0]
               for t, c in zip(trains, colors_t)]
for ax, ttl, yl in [(ax_risk, "⚠️  Collision Risk","Risk (0–1)"),
                     (ax_speed,"🚄  Velocities (km/h)","Speed")]:
    ax.set_title(ttl, fontsize=11, fontweight='bold')
    ax.set_ylabel(yl, fontsize=9)
    ax.legend(loc='upper left', fontsize=6, ncol=2)
    ax.grid(True, alpha=0.25)
ax_risk.set_ylim(0, 1)

# ── Track Occupancy Heatmap ───────────────────────────────────
OCC_WIN  = 40
occ_data = np.zeros((len(SEGMENTS), OCC_WIN))   # shape: (num_segments, time_window)

occ_im = ax_occ.imshow(
    occ_data, aspect='auto',
    cmap='RdYlGn_r',        # green=free, yellow=moderate, red=busy
    vmin=0, vmax=1,
    interpolation='nearest', origin='upper'
)
seg_ylabels = [f"{u}–{v}" for u,v in SEGMENTS]
ax_occ.set_yticks(range(len(SEGMENTS)))
ax_occ.set_yticklabels(seg_ylabels, fontsize=8)
ax_occ.set_xticks([0, OCC_WIN//2, OCC_WIN-1])
ax_occ.set_xticklabels([f"–{OCC_WIN}", f"–{OCC_WIN//2}", "now"], fontsize=8)
ax_occ.set_xlabel("← Older frames          Newer →", fontsize=8)
ax_occ.set_title("📡  Track Occupancy Heatmap\n(green = free  |  red = busy)",
                  fontsize=10, fontweight='bold')
cbar = plt.colorbar(occ_im, ax=ax_occ, fraction=0.05, pad=0.03)
cbar.set_label("Congestion level", fontsize=8)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(["0 free","0.5","1 busy"])

# ── Headway ───────────────────────────────────────────────────
hw_line, = ax_head.plot([], [], 'cyan', lw=2.5, label='Min Headway')
ax_head.axhline(CRITICAL_DIST, color='red',    ls='--', lw=2, label='Critical')
ax_head.axhline(WARNING_DIST,  color='orange', ls='--', lw=2, label='Warning')
ax_head.set_title("📏  Minimum Train Headway", fontsize=11, fontweight='bold')
ax_head.set_ylabel("Distance", fontsize=9); ax_head.set_xlabel("Frame", fontsize=9)
ax_head.legend(fontsize=8); ax_head.grid(True, alpha=0.25)
ax_head.set_ylim(0, 9)

# ── Brake bars ────────────────────────────────────────────────
brake_bars = ax_brake.barh(range(len(trains)), [0]*len(trains),
                            color='#FF5252', alpha=0.82, height=0.6)
ax_brake.set_yticks(range(len(trains)))
ax_brake.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_brake.set_xlim(0, 1.1)
ax_brake.set_title("🛑  Brake Intensity", fontsize=11, fontweight='bold')
ax_brake.set_xlabel("0 = None   →   1 = Full", fontsize=9)
ax_brake.grid(True, alpha=0.25, axis='x')
brake_txt = [ax_brake.text(0.02, i, '0.00', va='center', fontsize=8, color='white')
             for i in range(len(trains))]

# ── Reroute bars ──────────────────────────────────────────────
reroute_bars = ax_reroute.barh(range(len(trains)), [0]*len(trains),
                                color='#29B6F6', alpha=0.85, height=0.6)
ax_reroute.set_yticks(range(len(trains)))
ax_reroute.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_reroute.set_xlim(0, 12)
ax_reroute.set_title("🔀  Reroute Events", fontsize=11, fontweight='bold')
ax_reroute.set_xlabel("Times rerouted", fontsize=8)
ax_reroute.grid(True, alpha=0.25, axis='x')
reroute_txt = [ax_reroute.text(0.15, i, '0', va='center', fontsize=9,
                                color='white', fontweight='bold')
               for i in range(len(trains))]

ax_stats.axis('off')

# Histories
risk_hist  = {t.train_id: [] for t in trains}
speed_hist = {t.train_id: [] for t in trains}
hw_hist, ts = [], []

print("\nDashboard ready. Building animation …")

# ══════════════════════════════════════════════════════════════
# 8. ANIMATION UPDATE
# ══════════════════════════════════════════════════════════════
def update(frame):
    # ─ 1. occupancy ──────────────────────────────────────────
    update_occupancy(trains)

    # ─ 2. roll heatmap left, write new column ────────────────
    occ_data[:, :-1] = occ_data[:, 1:]
    for si, seg in enumerate(SEGMENTS):
        occ_data[si, -1] = seg_pred[seg]
    occ_im.set_data(occ_data)

    # ─ 3. colour track-load overlay on map ───────────────────
    for seg, ln in edge_lines.items():
        load = seg_pred[seg]
        ln.set_color(load_cmap(load))
        ln.set_alpha(0.20 + 0.70*load)

    # ─ 4. update trains ──────────────────────────────────────
    xs, ys, cols = [], [], []
    risks, speeds = [], []
    braking_count = 0

    for i, train in enumerate(trains):
        train.update()
        risk = compute_risk(train, trains)

        # Reroute ONLY if THIS train has a collision partner on the same edge
        ce = train.current_edge()
        if ce:
            for other in trains:
                if other.train_id == train.train_id:
                    continue
                oe = other.current_edge()
                if oe and (oe == ce or oe == (ce[1], ce[0])):
                    d = np.linalg.norm(
                        np.array(train.get_position()) - np.array(other.get_position())
                    )
                    if d < WARNING_DIST:
                        train.try_reroute(frame)
                        break

        risk_hist[train.train_id].append(risk)
        speed_hist[train.train_id].append(train.velocity)
        risks.append(risk); speeds.append(train.velocity)

        if train.is_braking:
            braking_count += 1
            brake_circles[i].set_center(train.get_position())
            brake_circles[i].set_visible(True)
        else:
            brake_circles[i].set_visible(False)

        if train.reroute_flash > 0:
            reroute_circles[i].set_center(train.get_position())
            reroute_circles[i].set_visible(True)
        else:
            reroute_circles[i].set_visible(False)

        if   risk > 0.70: train.color = "#FF1744"
        elif risk > 0.45: train.color = "#FF9800"
        elif risk > 0.20: train.color = "#FFEB3B"
        else:             train.color = "#4CAF50"

        x, y = train.get_position()
        xs.append(x); ys.append(y); cols.append(train.color)
        tlabels[i].set_position((x, y + 0.44))
        tlabels[i].set_text(train.train_id)

    train_scat.set_offsets(np.c_[xs, ys])
    train_scat.set_color(cols)

    # ─ 5. headway ────────────────────────────────────────────
    min_hw = min(
        np.linalg.norm(np.array(t1.get_position()) - np.array(t2.get_position()))
        for i, t1 in enumerate(trains) for t2 in trains[i+1:]
    )
    hw_hist.append(min_hw); ts.append(frame)
    hw_line.set_data(ts, hw_hist)
    ax_head.set_xlim(0, max(50, frame+1))

    # ─ 6. risk / speed plots ─────────────────────────────────
    for i, train in enumerate(trains):
        h = risk_hist[train.train_id]
        risk_lines[i].set_data(range(len(h)), h)
        s = speed_hist[train.train_id]
        speed_lines[i].set_data(range(len(s)), s)
    ax_risk.set_xlim(0,  max(50, frame+1))
    ax_speed.set_xlim(0, max(50, frame+1))
    if speeds:
        ax_speed.set_ylim(0, max(speeds+[5])*1.1)

    # ─ 7. brake bars ─────────────────────────────────────────
    for j, (bar, train) in enumerate(zip(brake_bars, trains)):
        bar.set_width(train.brake_intensity)
        brake_txt[j].set_x(train.brake_intensity + 0.01)
        brake_txt[j].set_text(f"{train.brake_intensity:.2f}")

    # ─ 8. reroute bars ───────────────────────────────────────
    max_r = max((t.reroute_count for t in trains), default=0)
    ax_reroute.set_xlim(0, max(12, max_r + 2))
    for j, (bar, train) in enumerate(zip(reroute_bars, trains)):
        bar.set_width(train.reroute_count)
        reroute_txt[j].set_x(train.reroute_count + 0.12)
        reroute_txt[j].set_text(str(train.reroute_count))

    # ─ 9. stats ──────────────────────────────────────────────
    ax_stats.clear(); ax_stats.axis('off')
    busy     = sum(1 for v in seg_pred.values() if v > REROUTE_THRESH)
    total_r  = sum(t.reroute_count for t in trains)
    status   = ("🔴 ALERT!"  if min_hw < CRITICAL_DIST else
                "🟡 Caution" if min_hw < WARNING_DIST  else
                "🟢 Normal")
    ax_stats.text(0.05, 0.5,
        f"  ⏱  Frame:         {frame}\n\n"
        f"  🚂 Trains:        {len(trains)}\n"
        f"  🛑 Braking:       {braking_count}\n"
        f"  🔀 Reroutes:      {total_r}\n\n"
        f"  📏 Min Headway:   {min_hw:.2f}\n"
        f"  📊 Avg Risk:      {np.mean(risks):.3f}\n"
        f"  ⚠️  Max Risk:      {max(risks):.3f}\n\n"
        f"  📡 Busy Segs:     {busy}/{len(SEGMENTS)}\n\n"
        f"  STATUS: {status}",
        fontsize=10, va='center', family='monospace', color='white',
        transform=ax_stats.transAxes)

    return ([train_scat, occ_im, hw_line]
            + list(edge_lines.values())
            + brake_circles + reroute_circles
            + risk_lines + speed_lines + tlabels)

# ══════════════════════════════════════════════════════════════
# 9. RUN
# ══════════════════════════════════════════════════════════════
ani = FuncAnimation(fig, update, frames=1200, interval=50, blit=False)
plt.close()

print("\n✅ Animation ready!\n")
print("Dashboard guide:")
print("  🟡 Gold squares     = Major Stations (trains MUST visit)")
print("  ⚫ Grey circles     = Minor Stations (skipped on congestion)")
print("  🎨 Train colour     = Green / Yellow / Orange / Red  (risk level)")
print("  🔴 Red ring         = Train is braking")
print("  🔵 Cyan dashed ring = Train just rerouted (flashes 18 frames)")
print("  🗺  Track overlay    = Green=free → Yellow → Red=congested")
print("  📡 Heatmap rows     = One row per segment, red = currently busy")
print("  🔀 Blue bars        = Reroute count per train (grows over time)")

HTML(ani.to_jshtml())

# ***Best working till now***

In [ ]:
"""
Railway Train Monitoring System
─────────────────────────────────────────────────────────────
FEATURES:
  ✅ Clean network — 4 major stations, deliberate crossing routes
  ✅ Signal lights (🔴🟡🟢) drawn ON map at each major station
  ✅ Head-on collision → both trains STOP visually (no teleport)
  ✅ Rerouting only at major stations, path shown on map
  ✅ Track occupancy heatmap, brake bars, reroute counters
─────────────────────────────────────────────────────────────
"""

import sys, os, warnings, logging, random, heapq, pickle, json
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, FancyArrowPatch
import networkx as nx
from IPython.display import HTML

warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.ERROR)
logging.getLogger('torch').setLevel(logging.ERROR)

# ══════════════════════════════════════════════════════════════
# 1. NETWORK  — simple, spacious, designed for visibility
#
#        W ────── mWC ────── C ────── mCE ────── E
#        |                   |                   |
#       mWS               mCS_n               mES
#        |                   |                   |
#        S ────── mSC ──────────────── mSE ───── E
#
#   4 Major stations:  W (West), C (Centre), E (East), S (South)
#   6 Minor waypoints: mWC, mCE, mWS, mCS_n, mES, mSC, mSE
#
#   Deliberate crossing routes:
#     T1: W→C→E          goes east
#     T2: E→C→W          goes west  — meets T1 head-on between C and E
#     T3: W→S→E          southern arc
#     T4: E→S→W          southern arc reverse — meets T3 head-on
#     T5: S→C→W          south to north then west
# ══════════════════════════════════════════════════════════════
G = nx.Graph()

pos = {
    # Major stations — well spaced for clarity
    "W":  (0,   6),   # MAJOR — West
    "C":  (6,   6),   # MAJOR — Centre
    "E":  (12,  6),   # MAJOR — East
    "S":  (6,   1),   # MAJOR — South

    # Minor waypoints — northern corridor W↔C↔E
    "mWC": (3,   6),  # minor — West-Centre mid
    "mCE": (9,   6),  # minor — Centre-East mid

    # Minor waypoints — southern arc W↔S↔E
    "mWS": (2,   2),  # minor — West-South
    "mSE": (10,  2),  # minor — South-East

    # Minor waypoints — cross connectors
    "mCS": (6,   3),  # minor — Centre-South mid
    "mNE": (10,  8),  # minor — North-East spur (alternate C→E northern)
    "mNW": (2,   8),  # minor — North-West spur (alternate W→C northern)
}

MAJOR_STATIONS = {"W", "C", "E", "S"}
MINOR_STATIONS = {"mWC", "mCE", "mWS", "mSE", "mCS", "mNE", "mNW"}

# Edge weights designed so:
#  - Main corridor (W-mWC-C-mCE-E) is fastest: weight 3 each
#  - Southern arc  (W-mWS-S-mSE-E) is slightly longer: weight 4 each
#  - Northern spurs are slowest: weight 5 — only used when main congested
weighted_edges = [
    # Main northern corridor
    ("W",   "mWC", 3), ("mWC", "C",   3),
    ("C",   "mCE", 3), ("mCE", "E",   3),

    # Southern arc
    ("W",   "mWS", 4), ("mWS", "S",   4),
    ("S",   "mSE", 4), ("mSE", "E",   4),

    # Centre ↔ South direct
    ("C",   "mCS", 3), ("mCS", "S",   3),

    # Northern spurs — alternate paths created by rerouting pressure
    ("W",   "mNW", 5), ("mNW", "C",   5),
    ("C",   "mNE", 5), ("mNE", "E",   5),
]
G.add_weighted_edges_from(weighted_edges)

# Canonical edge dict — both directions → one key
_CANONICAL = {}
for u, v in G.edges():
    _CANONICAL[(u, v)] = (u, v)
    _CANONICAL[(v, u)] = (u, v)

def canonical_edge(a, b):
    return _CANONICAL[(a, b)]

SEGMENTS = sorted({_CANONICAL[(u,v)] for u,v in G.edges()})

# ══════════════════════════════════════════════════════════════
# 2. SIGNAL STATE  — per major station, driven by occupancy
#    0 = Green  (clear)
#    1 = Yellow (train within 250m on approach)
#    2 = Red    (train within 100m — danger)
# ══════════════════════════════════════════════════════════════
# Signal for each MAJOR station — shows on map as coloured dot
station_signal = {s: 0 for s in MAJOR_STATIONS}

SIGNAL_RED_M    = 100   # metres
SIGNAL_YELLOW_M = 250

def update_signals(trains):
    """
    For each major station, find the closest incoming train.
    Set signal based on distance in metres (grid units × 100).
    """
    for station in MAJOR_STATIONS:
        min_dist = float("inf")
        for train in trains:
            # Euclidean grid distance × 100 → proxy metres
            sx, sy = pos[station]
            tx, ty = train.get_position()
            d = np.linalg.norm([tx-sx, ty-sy]) * 100
            if d < min_dist:
                min_dist = d
        if   min_dist < SIGNAL_RED_M:
            station_signal[station] = 2
        elif min_dist < SIGNAL_YELLOW_M:
            station_signal[station] = 1
        else:
            station_signal[station] = 0

# ══════════════════════════════════════════════════════════════
# 3. OCCUPANCY TRACKING
# ══════════════════════════════════════════════════════════════
HIST_LEN       = 40
MAX_ON_SEG          = 2   # 1 train=0.5, 2=1.0
PATH_REROUTE_THRESH = 0.25  # avg segment load to trigger reroute
REROUTE_THRESH = 0.70

seg_history = {s: deque([0]*HIST_LEN, maxlen=HIST_LEN) for s in SEGMENTS}
seg_load    = {s: 0   for s in SEGMENTS}
seg_pred    = {s: 0.0 for s in SEGMENTS}

def update_occupancy(trains):
    for s in SEGMENTS:
        seg_load[s] = 0
    for train in trains:
        e = train.current_edge()
        if e:
            s = canonical_edge(*e)
            seg_load[s] += 1
    for s in SEGMENTS:
        seg_history[s].append(seg_load[s])
        recent      = list(seg_history[s])[-10:]
        avg         = np.mean(recent) / MAX_ON_SEG
        curr        = seg_load[s]    / MAX_ON_SEG
        seg_pred[s] = float(np.clip(0.6*avg + 0.4*curr, 0.0, 1.0))

# ══════════════════════════════════════════════════════════════
# 4. DIJKSTRA + ROUTE PLANNING
# ══════════════════════════════════════════════════════════════
CONGESTION_PENALTY = 15.0

def build_cost_graph():
    cost = {}
    for u, v, d in G.edges(data=True):
        base    = d.get("weight", 3)
        s       = canonical_edge(u, v)
        penalty = CONGESTION_PENALTY * seg_pred[s] if seg_pred[s] > 0.40 else 0
        w = base + penalty
        cost.setdefault(u, {})[v] = w
        cost.setdefault(v, {})[u] = w
    return cost

def dijkstra(graph, src, dst):
    dist = {src: 0}
    prev = {}
    pq   = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, float("inf")):
            continue
        if u == dst:
            break
        for v, w in graph.get(u, {}).items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    path, cur = [], dst
    while cur != src:
        path.append(cur)
        cur = prev.get(cur)
        if cur is None:
            return None
    path.append(src)
    return list(reversed(path))

def mandatory_route(waypoints):
    cg   = build_cost_graph()
    full = []
    for i in range(len(waypoints)-1):
        seg = dijkstra(cg, waypoints[i], waypoints[i+1])
        if seg is None:
            return None
        full = full[:-1] + seg if full else seg
    return full

# ══════════════════════════════════════════════════════════════
# 5. TRAIN CLASS
# ══════════════════════════════════════════════════════════════
TRAIN_MANDATORY = {
    # T1 & T2: head-on on northern corridor (mWC or mCE segment)
    "T1": ["W", "C", "E"],    # Express:  west → centre → east
    "T2": ["E", "C", "W"],    # Regional: east → centre → west  ← head-on T1
    # T3 & T6: head-on on S↔C segment (mCS)
    "T3": ["S", "C", "W"],    # Regional: south → centre → west
    "T6": ["C", "S", "E"],    # Freight:  centre → south → east ← head-on T3
    # T4 & T5: rerouting showcase — both start W, different paths
    "T4": ["W", "S", "E"],    # Regional: west → south → east
    "T5": ["C", "W", "S"],    # Freight:  centre → west → south (unique path)
}

# Priority: 1=Express (never rerouted, others yield)
#           2=Regional (rerouted only when needed)
#           3=Freight  (rerouted first — lowest priority)
TRAIN_PRIORITY = {
    "T1": 1,   # Express  — never rerouted
    "T2": 2,   # Regional
    "T3": 2,   # Regional
    "T4": 2,   # Regional
    "T5": 3,   # Freight
    "T6": 3,   # Freight
}

class Train2D:
    def __init__(self, tid, speed_range=(0.010, 0.020)):
        self.train_id        = tid
        self.mandatory       = TRAIN_MANDATORY[tid]
        self.path            = mandatory_route(self.mandatory) or self.mandatory[:]
        self.edge_index      = 0
        self.progress        = random.uniform(0, 0.10)
        self.base_speed      = random.uniform(*speed_range)
        self.speed           = self.base_speed
        self.color           = "#4CAF50"
        self.history         = deque(maxlen=50)
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.velocity        = 0.0
        self.collision_risk  = 0.0
        self.reroute_count   = 0
        self.last_reroute    = -9999
        self.reroute_flash   = 0
        # Priority level from TRAIN_PRIORITY — lower = higher priority
        self.priority        = TRAIN_PRIORITY.get(tid, 3)
        # HEAD-ON STOP: when two trains face each other and can't reroute
        self.stopped_headon  = False
        self.headon_timer    = 0     # frames to stay stopped

    def current_edge(self):
        if self.edge_index >= len(self.path)-1:
            return None
        return (self.path[self.edge_index], self.path[self.edge_index+1])

    def apply_brake(self, intensity):
        self.is_braking      = True
        self.brake_intensity = intensity
        self.speed           = self.base_speed * (1 - intensity*0.8)

    def full_stop(self, frames=40):
        """Completely stop train for given number of frames (head-on scenario)."""
        self.stopped_headon  = True
        self.headon_timer    = frames
        self.speed           = 0.0
        self.brake_intensity = 1.0
        self.is_braking      = True

    def release_brake(self):
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.speed           = self.base_speed

    def update(self):
        # Decrement head-on stop timer
        if self.stopped_headon:
            self.headon_timer -= 1
            if self.headon_timer <= 0:
                self.stopped_headon  = False
                self.brake_intensity = 0.0
                self.is_braking      = False
                self.speed           = self.base_speed
            return   # don't move while stopped

        self.progress += self.speed
        if self.progress >= 1:
            self.progress   = 0
            self.edge_index = (self.edge_index+1) % max(1, len(self.path)-1)
        if self.reroute_flash > 0:
            self.reroute_flash -= 1

    def get_position(self):
        if self.edge_index >= len(self.path)-1:
            return pos[self.path[-1]]
        x1,y1 = pos[self.path[self.edge_index]]
        x2,y2 = pos[self.path[self.edge_index+1]]
        return (x1+(x2-x1)*self.progress, y1+(y2-y1)*self.progress)

    def _score_path(self, p):
        total = 0.0
        for i in range(len(p)-1):
            s = canonical_edge(p[i], p[i+1])
            total += seg_pred.get(s, 0.0)
        return total

    def choose_best_path(self, src, dst):
        """Enumerate all simple paths src→dst, return least-congested."""
        try:
            all_paths = list(nx.all_simple_paths(G, src, dst, cutoff=6))
        except Exception:
            all_paths = []
        visited          = set(self.path[:self.edge_index+1])
        unvisited_majors = set(self.mandatory) - visited - {src, dst}
        valid = [p for p in all_paths
                 if not set(p[1:-1]).intersection(unvisited_majors)]
        if not valid:
            return dijkstra(build_cost_graph(), src, dst)
        return min(valid, key=self._score_path)

    def plan_from_major(self, frame):
        """
        At a major station: pick least-congested path to next mandatory stop.
        Only reroutes when current path is genuinely congested AND a better
        alternative exists (>30% improvement).
        """
        if frame - self.last_reroute < 30:
            return False
        current_node = self.path[self.edge_index]
        if current_node not in MAJOR_STATIONS:
            return False
        visited_majors = {n for n in self.path[:self.edge_index+1] if n in MAJOR_STATIONS}
        remaining = [m for m in self.mandatory if m not in visited_majors]
        if not remaining:
            return False

        next_major = remaining[0]
        # Score current planned path
        try:
            nm_idx      = self.path[self.edge_index:].index(next_major)
            current_seg = self.path[self.edge_index: self.edge_index+nm_idx+1]
        except ValueError:
            current_seg = []
        current_score = self._score_path(current_seg)
        # PATH_REROUTE_THRESH is per-segment average — reroute if path is noticeably loaded
        path_len = max(len(current_seg)-1, 1)
        avg_load = current_score / path_len
        if avg_load < PATH_REROUTE_THRESH:
            return False   # path clear — no reroute needed

        best_path  = self.choose_best_path(current_node, next_major)
        if best_path is None:
            return False
        best_score = self._score_path(best_path)
        if best_score >= current_score * 0.90:
            return False   # not worth switching (only skip if < 10% better)

        after = remaining[1:]
        tail  = mandatory_route([next_major]+after) if after else []
        new_suffix = best_path + (tail[1:] if tail else [])
        if new_suffix == self.path[self.edge_index:]:
            return False

        self.path          = self.path[:self.edge_index] + new_suffix
        self.progress      = 0
        self.reroute_count += 1
        self.last_reroute  = frame
        self.reroute_flash = 25
        return True

# ══════════════════════════════════════════════════════════════
# 6. HEAD-ON DETECTION
#    If two trains are on the SAME edge in OPPOSITE directions
#    AND close enough that braking won't help, both full-stop.
#    No teleporting — they sit nose-to-nose until timer expires.
# ══════════════════════════════════════════════════════════════
HEAD_ON_DIST   = 1.20   # grid units — stop well before contact (visual gap visible)
HEAD_ON_FRAMES = 60     # frames to stay stopped before resuming

def detect_and_handle_headon(trains):
    handled = set()
    for i, t1 in enumerate(trains):
        if t1.stopped_headon:
            continue
        e1 = t1.current_edge()
        if e1 is None:
            continue
        for t2 in trains[i+1:]:
            if t2.stopped_headon:
                continue
            e2 = t2.current_edge()
            if e2 is None:
                continue
            # Head-on: same physical edge, opposite traversal direction
            if e2 == (e1[1], e1[0]):
                d = np.linalg.norm(
                    np.array(t1.get_position()) - np.array(t2.get_position())
                )
                if d < HEAD_ON_DIST and (t1.train_id, t2.train_id) not in handled:
                    t1.full_stop(HEAD_ON_FRAMES)
                    t2.full_stop(HEAD_ON_FRAMES)
                    handled.add((t1.train_id, t2.train_id))
                    handled.add((t2.train_id, t1.train_id))

# ══════════════════════════════════════════════════════════════
# 7. COLLISION RISK MODEL
# ══════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

risk_model = model.to(device)
risk_model.eval()

try:
    with open("scaler.pkl","rb") as f:
        _scaler = pickle.load(f)
except FileNotFoundError:
    _scaler = None

try:
    with open("model_config.json") as f:
        _cfg = json.load(f)
    _THRESHOLD = _cfg.get("optimal_threshold", 0.5)
except FileNotFoundError:
    _THRESHOLD = 0.5

_SEG_PROPS = {
    0: {"gradient":-0.005,"curvature":0.005,"speed_limit":34.6,"friction":0.88},
    1: {"gradient":-0.014,"curvature":0.001,"speed_limit":21.2,"friction":0.96},
    2: {"gradient": 0.004,"curvature":0.004,"speed_limit":20.4,"friction":0.99},
    3: {"gradient": 0.013,"curvature":0.001,"speed_limit":23.6,"friction":0.76},
    4: {"gradient":-0.008,"curvature":0.003,"speed_limit":28.6,"friction":0.79},
    5: {"gradient": 0.004,"curvature":0.001,"speed_limit":25.8,"friction":0.81},
    6: {"gradient":-0.002,"curvature":0.004,"speed_limit":24.0,"friction":0.85},
    7: {"gradient": 0.004,"curvature":0.000,"speed_limit":32.2,"friction":0.75},
}

CRITICAL_DIST = 0.35
WARNING_DIST  = 0.70
SAFE_DIST     = 1.50
SEQ_LEN       = 10

def compute_risk(train, others, frame=0):
    min_headway = 10000.0
    ce = train.current_edge()
    if ce is None:
        return 0.0

    for other in others:
        if other.train_id == train.train_id:
            continue
        oe = other.current_edge()
        # Same edge either direction
        if oe and (oe == ce or oe == (ce[1], ce[0])):
            d = np.linalg.norm(
                np.array(train.get_position()) - np.array(other.get_position())
            )
            min_headway = min(min_headway, d*100)

    velocity     = train.speed * 100.0
    acceleration = (train.speed - train.base_speed) * 100.0
    headway      = float(np.clip(min_headway, 0, 500))

    seg_idx  = train.edge_index % len(_SEG_PROPS)
    props    = _SEG_PROPS[seg_idx]

    if   min_headway < 100:  signal_state = 2.0
    elif min_headway < 250:  signal_state = 1.0
    else:                    signal_state = 0.0

    raw = np.array([
        velocity, acceleration, headway,
        props["gradient"], props["curvature"], props["speed_limit"],
        signal_state, props["friction"],
    ], dtype=np.float32)

    if _scaler is not None:
        feat = _scaler.transform(raw.reshape(1,-1))[0].astype(np.float32)
    else:
        feat = raw

    train.history.append(feat)

    if len(train.history) >= SEQ_LEN:
        seq = np.array(list(train.history)[-SEQ_LEN:])
        t   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            prob = torch.sigmoid(risk_model(t)).item()
    else:
        prob = 1.0 if min_headway < CRITICAL_DIST*100 else (
               0.5 if min_headway < WARNING_DIST*100  else 0.1)

    train.velocity       = velocity
    train.collision_risk = prob

    dist_m = min_headway
    if   dist_m < CRITICAL_DIST*100:
        train.apply_brake(min(1.0,(CRITICAL_DIST*100-dist_m)/(CRITICAL_DIST*100)))
    elif dist_m < WARNING_DIST*100:
        train.apply_brake(0.5)
    elif dist_m < SAFE_DIST*100:
        train.apply_brake(0.2)
    elif not train.stopped_headon:
        train.release_brake()

    return prob

# ══════════════════════════════════════════════════════════════
# 8. SEQUENTIAL REROUTING
# ══════════════════════════════════════════════════════════════
def reroute_trains_sequentially(trains, frame):
    """
    Exactly ONE train reroutes per frame in priority order.
    Freight (3) reroutes before Regional (2). Express (1) never reroutes.
    After the chosen train commits, occupancy is updated immediately so
    the next frame's candidate sees the reserved path and picks differently.
    """
    candidates = []
    for train in trains:
        if train.stopped_headon:                              continue
        if train.priority == 1:                              continue
        if train.path[train.edge_index] not in MAJOR_STATIONS: continue
        if train.progress >= 0.20:                           continue
        if frame - train.last_reroute < 30:                  continue

        # Find next unvisited mandatory major station
        visited_majors = {n for n in train.path[:train.edge_index+1]
                          if n in MAJOR_STATIONS}
        remaining = [m for m in train.mandatory if m not in visited_majors]
        if not remaining:
            continue
        next_major = remaining[0]
        try:
            nm_idx      = train.path[train.edge_index:].index(next_major)
            current_seg = train.path[train.edge_index: train.edge_index+nm_idx+1]
        except ValueError:
            continue
        avg_load = train._score_path(current_seg) / max(len(current_seg)-1, 1)
        if avg_load < PATH_REROUTE_THRESH:
            continue
        candidates.append((train.priority, id(train), train, remaining, next_major, current_seg))

    if not candidates:
        return

    # Highest priority number (freight=3) goes first — sort descending by priority
    candidates.sort(key=lambda x: -x[0])
    _, _, chosen, remaining, next_major, current_seg = candidates[0]

    # Find best alternative path
    best_path = chosen.choose_best_path(chosen.path[chosen.edge_index], next_major)
    if best_path is None:
        return

    current_score = chosen._score_path(current_seg)
    best_score    = chosen._score_path(best_path)

    # Only switch if meaningfully better (at least 10% improvement)
    if best_score >= current_score * 0.90:
        return

    after      = remaining[1:]
    tail       = mandatory_route([next_major] + after) if after else []
    new_suffix = best_path + (tail[1:] if tail else [])
    if new_suffix == chosen.path[chosen.edge_index:]:
        return

    # Commit — exactly one reroute this frame
    chosen.path          = chosen.path[:chosen.edge_index] + new_suffix
    chosen.progress      = 0
    chosen.reroute_count += 1
    chosen.last_reroute  = frame
    chosen.reroute_flash = 25
    update_occupancy(trains)   # reserve path immediately for next frame

# ══════════════════════════════════════════════════════════════
# 9. CREATE TRAINS
# ══════════════════════════════════════════════════════════════
speed_ranges = {
    "T1":(0.013,0.018),  # Express — fastest
    "T2":(0.011,0.016),  # Regional
    "T3":(0.011,0.016),  # Regional
    "T4":(0.010,0.015),  # Regional
    "T5":(0.009,0.014),  # Freight — slowest
    "T6":(0.009,0.014),  # Freight — slowest
}
trains = [Train2D(tid, speed_ranges[tid]) for tid in TRAIN_MANDATORY]

print(f"✅ {len(trains)} trains | {len(G.nodes())} stations | {len(G.edges())} segments | device={device}")

# ══════════════════════════════════════════════════════════════
# 10. DASHBOARD
# ══════════════════════════════════════════════════════════════
plt.style.use('dark_background')
fig = plt.figure(figsize=(28, 16), dpi=90)
gs  = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.38)

ax_net     = fig.add_subplot(gs[:2, :2])
ax_risk    = fig.add_subplot(gs[0,  2])
ax_speed   = fig.add_subplot(gs[1,  2])
ax_occ     = fig.add_subplot(gs[0,  3])
ax_head    = fig.add_subplot(gs[2,  0])
ax_brake   = fig.add_subplot(gs[2,  1])
ax_reroute = fig.add_subplot(gs[2,  2])
ax_stats   = fig.add_subplot(gs[2,  3])

# ── Network base ──────────────────────────────────────────────
nx.draw_networkx_edges(G, pos=pos, ax=ax_net,
                       edge_color='#37474F', width=7, alpha=0.6)

# Major stations — gold squares
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MAJOR_STATIONS),
                       node_color='#FFD700', node_size=2200,
                       node_shape='s', alpha=0.95)
# Minor waypoints — grey circles
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MINOR_STATIONS),
                       node_color='#546E7A', node_size=600,
                       node_shape='o', alpha=0.80)
nx.draw_networkx_labels(G, pos=pos, ax=ax_net,
                        font_color='white', font_weight='bold', font_size=12)

# ── Signal light markers on major stations ────────────────────
# Small coloured circles drawn ABOVE each major station square
# Updated every frame to reflect station_signal state
SIGNAL_COLORS = {0: '#00E676', 1: '#FFD740', 2: '#FF1744'}  # green, yellow, red
signal_dots = {}
for station in MAJOR_STATIONS:
    sx, sy = pos[station]
    dot = ax_net.scatter(sx, sy+0.55, s=220, zorder=10,
                         color=SIGNAL_COLORS[0],
                         edgecolors='white', linewidths=1.5)
    signal_dots[station] = dot

# Signal legend
sig_patches = [
    mpatches.Patch(color='#00E676', label='Signal: Green (clear)'),
    mpatches.Patch(color='#FFD740', label='Signal: Yellow (caution)'),
    mpatches.Patch(color='#FF1744', label='Signal: Red (danger)'),
    mpatches.Patch(color='#FFD700', label='Major Station'),
    mpatches.Patch(color='#546E7A', label='Minor Waypoint'),
]
ax_net.legend(handles=sig_patches, loc='upper right', fontsize=8,
              framealpha=0.7)

# ── Track load overlays — coloured lines per segment ─────────
load_cmap  = plt.cm.RdYlGn_r
edge_lines = {}
for (u,v) in SEGMENTS:
    x1,y1 = pos[u]; x2,y2 = pos[v]
    ln, = ax_net.plot([x1,x2],[y1,y2], lw=9, alpha=0.0, zorder=2,
                      solid_capstyle='round')
    edge_lines[(u,v)] = ln

# ── Train scatter + indicators ────────────────────────────────
colors_t     = ['#00E5FF','#FF6D00','#69FF47','#FF4DFF','#FFD700','#FF1744']
train_scat   = ax_net.scatter([], [], s=520, zorder=6,
                               edgecolors='white', linewidths=2.5)

# Brake indicator — red pulsing ring
brake_circles = [Circle((0,0), 0.38, fill=False, edgecolor='#FF1744',
                          linewidth=4, visible=False, zorder=8) for _ in trains]
# Reroute flash — cyan dashed ring
reroute_circles = [Circle((0,0), 0.52, fill=False, edgecolor='#00E5FF',
                            linewidth=3, linestyle='--', visible=False, zorder=8)
                   for _ in trains]
# Head-on stop — white solid ring (larger)
headon_circles = [Circle((0,0), 0.62, fill=False, edgecolor='#FFFFFF',
                           linewidth=4, visible=False, zorder=8) for _ in trains]

for c in brake_circles + reroute_circles + headon_circles:
    ax_net.add_patch(c)

tlabels = [ax_net.text(0,0,'', fontsize=9, color='white', ha='center',
                        va='bottom', fontweight='bold', zorder=9)
           for _ in trains]

ax_net.set_title("🚂  Railway Network — Signals, Braking, Rerouting",
                  fontsize=14, fontweight='bold', pad=12)
ax_net.set_xlim(-1.5, 14); ax_net.set_ylim(-1, 10.5); ax_net.axis('off')

# ── Risk / Speed lines ────────────────────────────────────────
risk_lines  = [ax_risk.plot([],[],  color=c, lw=2, label=t.train_id)[0]
               for t,c in zip(trains,colors_t)]
speed_lines = [ax_speed.plot([],[], color=c, lw=2, label=t.train_id)[0]
               for t,c in zip(trains,colors_t)]
for ax, ttl, yl in [(ax_risk,"⚠️  Collision Risk","Risk (0–1)"),
                     (ax_speed,"🚄  Velocities","Speed")]:
    ax.set_title(ttl, fontsize=11, fontweight='bold')
    ax.set_ylabel(yl, fontsize=9)
    ax.legend(loc='upper left', fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
ax_risk.set_ylim(0, 1)

# ── Occupancy heatmap ─────────────────────────────────────────
OCC_WIN  = 40
occ_data = np.zeros((len(SEGMENTS), OCC_WIN))
occ_im   = ax_occ.imshow(occ_data, aspect='auto', cmap='RdYlGn_r',
                          vmin=0, vmax=1, interpolation='nearest', origin='upper')
ax_occ.set_yticks(range(len(SEGMENTS)))
ax_occ.set_yticklabels([f"{u}–{v}" for u,v in SEGMENTS], fontsize=7)
ax_occ.set_xticks([0, OCC_WIN//2, OCC_WIN-1])
ax_occ.set_xticklabels([f"–{OCC_WIN}","",  "now"], fontsize=8)
ax_occ.set_title("📡  Track Occupancy\n(green=free | red=busy)",
                  fontsize=10, fontweight='bold')
plt.colorbar(occ_im, ax=ax_occ, fraction=0.05, pad=0.03).set_label("Congestion", fontsize=8)

# ── Headway ───────────────────────────────────────────────────
hw_line, = ax_head.plot([],[], 'cyan', lw=2.5, label='Min Headway')
ax_head.axhline(CRITICAL_DIST, color='red',    ls='--', lw=2, label='Critical/Stop')
ax_head.axhline(WARNING_DIST,  color='orange', ls='--', lw=2, label='Warning')
ax_head.set_title("📏  Minimum Headway (grid units)", fontsize=11, fontweight='bold')
ax_head.set_ylabel("Distance", fontsize=9); ax_head.set_xlabel("Frame", fontsize=9)
ax_head.legend(fontsize=8); ax_head.grid(True, alpha=0.25)
ax_head.set_ylim(0, 8)

# ── Brake bars ────────────────────────────────────────────────
brake_bars = ax_brake.barh(range(len(trains)), [0]*len(trains),
                            color='#FF5252', alpha=0.82, height=0.6)
ax_brake.set_yticks(range(len(trains)))
ax_brake.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_brake.set_xlim(0, 1.1)
ax_brake.set_title("🛑  Brake Intensity", fontsize=11, fontweight='bold')
ax_brake.set_xlabel("0 = None  →  1 = Full Stop", fontsize=9)
ax_brake.grid(True, alpha=0.25, axis='x')
brake_txt = [ax_brake.text(0.02, i, '0.00', va='center', fontsize=8, color='white')
             for i in range(len(trains))]

# ── Reroute bars ──────────────────────────────────────────────
reroute_bars = ax_reroute.barh(range(len(trains)), [0]*len(trains),
                                color='#29B6F6', alpha=0.85, height=0.6)
ax_reroute.set_yticks(range(len(trains)))
ax_reroute.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_reroute.set_xlim(0, 12)
ax_reroute.set_title("🔀  Reroute Events", fontsize=11, fontweight='bold')
ax_reroute.set_xlabel("Times rerouted", fontsize=9)
ax_reroute.grid(True, alpha=0.25, axis='x')
reroute_txt = [ax_reroute.text(0.15, i, '0', va='center', fontsize=9,
                                color='white', fontweight='bold')
               for i in range(len(trains))]

ax_stats.axis('off')

risk_hist  = {t.train_id: [] for t in trains}
speed_hist = {t.train_id: [] for t in trains}
hw_hist, ts = [], []

print("\nDashboard ready. Building animation …")

# ══════════════════════════════════════════════════════════════
# 11. ANIMATION
# ══════════════════════════════════════════════════════════════
def update(frame):
    # ─ 1. occupancy + signals ─────────────────────────────────
    update_occupancy(trains)
    update_signals(trains)

    # ─ 2. heatmap ─────────────────────────────────────────────
    occ_data[:, :-1] = occ_data[:, 1:]
    for si, seg in enumerate(SEGMENTS):
        occ_data[si, -1] = seg_pred[seg]
    occ_im.set_data(occ_data)

    # ─ 3. track load colours ──────────────────────────────────
    for seg, ln in edge_lines.items():
        load = seg_pred[seg]
        ln.set_color(load_cmap(load))
        ln.set_alpha(0.15 + 0.75*load)

    # ─ 4. update signal dots on map ───────────────────────────
    for station, dot in signal_dots.items():
        dot.set_color(SIGNAL_COLORS[station_signal[station]])

    # ─ 5. head-on detection ───────────────────────────────────
    detect_and_handle_headon(trains)

    # ─ 6. update trains ───────────────────────────────────────
    xs, ys, cols = [], [], []
    risks, speeds = [], []
    braking_count = 0

    for i, train in enumerate(trains):
        train.update()
        risk = compute_risk(train, trains, frame)

        # Rerouting handled sequentially outside this loop (see below)

        risk_hist[train.train_id].append(risk)
        speed_hist[train.train_id].append(train.velocity)
        risks.append(risk); speeds.append(train.velocity)

        x, y = train.get_position()

        # Head-on stop ring — white solid, largest
        if train.stopped_headon:
            headon_circles[i].set_center((x, y))
            headon_circles[i].set_visible(True)
            brake_circles[i].set_visible(False)
        else:
            headon_circles[i].set_visible(False)

        # Brake ring — red
        if train.is_braking and not train.stopped_headon:
            braking_count += 1
            brake_circles[i].set_center((x, y))
            brake_circles[i].set_visible(True)
        else:
            if not train.stopped_headon:
                brake_circles[i].set_visible(False)

        # Reroute flash ring — cyan dashed
        if train.reroute_flash > 0:
            reroute_circles[i].set_center((x, y))
            reroute_circles[i].set_visible(True)
        else:
            reroute_circles[i].set_visible(False)

        # Train colour by risk level
        if   train.stopped_headon: train.color = "#FFFFFF"   # white = stopped
        elif risk > _THRESHOLD:    train.color = "#FF1744"   # red
        elif risk > 0.40:          train.color = "#FF9800"   # orange
        elif risk > 0.20:          train.color = "#FFEB3B"   # yellow
        else:                      train.color = "#4CAF50"   # green

        xs.append(x); ys.append(y); cols.append(train.color)
        tlabels[i].set_position((x, y+0.48))
        prio_sym = "★" if train.priority == 1 else ("◆" if train.priority == 2 else "●")
        tlabels[i].set_text(f"{train.train_id}{prio_sym}")

        if train.is_braking and not train.stopped_headon:
            braking_count += 1

    train_scat.set_offsets(np.c_[xs, ys])
    train_scat.set_color(cols)

    # ─ 6b. sequential rerouting — one train per frame, priority order ──
    reroute_trains_sequentially(trains, frame)

    # ─ 7. headway ─────────────────────────────────────────────
    min_hw = min(
        np.linalg.norm(np.array(t1.get_position()) - np.array(t2.get_position()))
        for ii, t1 in enumerate(trains) for t2 in trains[ii+1:]
    )
    hw_hist.append(min_hw); ts.append(frame)
    hw_line.set_data(ts, hw_hist)
    ax_head.set_xlim(0, max(50, frame+1))

    # ─ 8. risk / speed plots ──────────────────────────────────
    for i, train in enumerate(trains):
        h = risk_hist[train.train_id]
        risk_lines[i].set_data(range(len(h)), h)
        s = speed_hist[train.train_id]
        speed_lines[i].set_data(range(len(s)), s)
    ax_risk.set_xlim(0,  max(50, frame+1))
    ax_speed.set_xlim(0, max(50, frame+1))
    if speeds:
        ax_speed.set_ylim(0, max(speeds+[1])*1.15)

    # ─ 9. brake bars ──────────────────────────────────────────
    for j, (bar, train) in enumerate(zip(brake_bars, trains)):
        bar.set_width(train.brake_intensity)
        brake_txt[j].set_x(train.brake_intensity + 0.01)
        brake_txt[j].set_text(f"{train.brake_intensity:.2f}")

    # ─ 10. reroute bars ───────────────────────────────────────
    max_r = max((t.reroute_count for t in trains), default=0)
    ax_reroute.set_xlim(0, max(12, max_r+2))
    for j, (bar, train) in enumerate(zip(reroute_bars, trains)):
        bar.set_width(train.reroute_count)
        reroute_txt[j].set_x(train.reroute_count + 0.12)
        reroute_txt[j].set_text(str(train.reroute_count))

    # ─ 11. stats panel ────────────────────────────────────────
    ax_stats.clear(); ax_stats.axis('off')
    stopped  = sum(1 for t in trains if t.stopped_headon)
    total_r  = sum(t.reroute_count for t in trains)
    sig_str  = " ".join(
        f"{s}:{'🔴' if station_signal[s]==2 else '🟡' if station_signal[s]==1 else '🟢'}"
        for s in sorted(MAJOR_STATIONS)
    )
    status   = ("🔴 HEAD-ON STOP" if stopped > 0 else
                "🔴 ALERT!"       if min_hw < CRITICAL_DIST else
                "🟡 Caution"      if min_hw < WARNING_DIST  else
                "🟢 Normal")
    ax_stats.text(0.04, 0.5,
        f"  ⏱  Frame:       {frame}\n\n"
        f"  🚂 Trains:      {len(trains)}\n"
        f"  🛑 Braking:     {braking_count}\n"
        f"  ⛔ Head-on Stop:{stopped}\n"
        f"  🔀 Reroutes:    {total_r}\n\n"
        f"  📏 Min Headway: {min_hw:.2f}\n"
        f"  📊 Avg Risk:    {np.mean(risks):.3f}\n"
        f"  ⚠️  Max Risk:    {max(risks):.3f}\n\n"
        f"  Signals:\n  {sig_str}\n\n"
        f"  {status}",
        fontsize=9, va='center', family='monospace', color='white',
        transform=ax_stats.transAxes)

    return ([train_scat, occ_im, hw_line]
            + list(edge_lines.values())
            + list(signal_dots.values())
            + brake_circles + reroute_circles + headon_circles
            + risk_lines + speed_lines + tlabels)

# ══════════════════════════════════════════════════════════════
# 12. RUN
# ══════════════════════════════════════════════════════════════
ani = FuncAnimation(fig, update, frames=1200, interval=50, blit=False)
plt.close()

HTML(ani.to_jshtml())

Output hidden; open in https://colab.research.google.com to view.

# **Final**

---



In [ ]:
"""
Railway Train Monitoring System
─────────────────────────────────────────────────────────────
FEATURES:
  ✅ Clean network — 4 major stations, deliberate crossing routes
  ✅ Signal lights (🔴🟡🟢) drawn ON map at each major station
  ✅ Head-on collision → both trains STOP visually (no teleport)
  ✅ Rerouting only at major stations, path shown on map
  ✅ Track occupancy heatmap, brake bars, reroute counters
─────────────────────────────────────────────────────────────
"""

import sys, os, warnings, logging, random, heapq, pickle, json
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle, FancyArrowPatch
import networkx as nx
from IPython.display import HTML

warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.ERROR)
logging.getLogger('torch').setLevel(logging.ERROR)


G = nx.Graph()

pos = {
    # Major stations — well spaced for clarity
    "W":  (0,   6),   # MAJOR — West
    "C":  (6,   6),   # MAJOR — Centre
    "E":  (12,  6),   # MAJOR — East
    "S":  (6,   1),   # MAJOR — South

    # Minor waypoints — northern corridor W↔C↔E
    "mWC": (3,   6),  # minor — West-Centre mid
    "mCE": (9,   6),  # minor — Centre-East mid

    # Minor waypoints — southern arc W↔S↔E
    "mWS": (2,   2),  # minor — West-South
    "mSE": (10,  2),  # minor — South-East

    # Minor waypoints — cross connectors
    "mCS": (6,   3),  # minor — Centre-South mid
    "mNE": (10,  8),  # minor — North-East spur (alternate C→E northern)
    "mNW": (2,   8),  # minor — North-West spur (alternate W→C northern)
}

MAJOR_STATIONS = {"W", "C", "E", "S"}
MINOR_STATIONS = {"mWC", "mCE", "mWS", "mSE", "mCS", "mNE", "mNW"}

# Edge weights designed so:
#  - Main corridor (W-mWC-C-mCE-E) is fastest: weight 3 each
#  - Southern arc  (W-mWS-S-mSE-E) is slightly longer: weight 4 each
#  - Northern spurs are slowest: weight 5 — only used when main congested
weighted_edges = [
    # Main northern corridor
    ("W",   "mWC", 3), ("mWC", "C",   3),
    ("C",   "mCE", 3), ("mCE", "E",   3),

    # Southern arc
    ("W",   "mWS", 4), ("mWS", "S",   4),
    ("S",   "mSE", 4), ("mSE", "E",   4),

    # Centre ↔ South direct
    ("C",   "mCS", 3), ("mCS", "S",   3),

    # Northern spurs — alternate paths created by rerouting pressure
    ("W",   "mNW", 5), ("mNW", "C",   5),
    ("C",   "mNE", 5), ("mNE", "E",   5),
]
G.add_weighted_edges_from(weighted_edges)


_CANONICAL = {}
for u, v in G.edges():
    _CANONICAL[(u, v)] = (u, v)
    _CANONICAL[(v, u)] = (u, v)

def canonical_edge(a, b):
    return _CANONICAL[(a, b)]

SEGMENTS = sorted({_CANONICAL[(u,v)] for u,v in G.edges()})

# ══════════════════════════════════════════════════════════════
# 2. SIGNAL STATE  — per major station, driven by occupancy
#    0 = Green  (clear)
#    1 = Yellow (train within 250m on approach)
#    2 = Red    (train within 100m — danger)
# ══════════════════════════════════════════════════════════════
# Signal for each MAJOR station — shows on map as coloured dot
station_signal = {s: 0 for s in MAJOR_STATIONS}

SIGNAL_RED_M    = 100   # metres
SIGNAL_YELLOW_M = 250

def update_signals(trains):
    """
    For each major station, find the closest incoming train.
    Set signal based on distance in metres (grid units × 100).
    """
    for station in MAJOR_STATIONS:
        min_dist = float("inf")
        for train in trains:
            # Euclidean grid distance × 100 → proxy metres
            sx, sy = pos[station]
            tx, ty = train.get_position()
            d = np.linalg.norm([tx-sx, ty-sy]) * 100
            if d < min_dist:
                min_dist = d
        if   min_dist < SIGNAL_RED_M:
            station_signal[station] = 2
        elif min_dist < SIGNAL_YELLOW_M:
            station_signal[station] = 1
        else:
            station_signal[station] = 0


HIST_LEN       = 40
MAX_ON_SEG          = 2   # 1 train=0.5, 2=1.0
PATH_REROUTE_THRESH = 0.25  # avg segment load to trigger reroute
REROUTE_THRESH = 0.70

seg_history = {s: deque([0]*HIST_LEN, maxlen=HIST_LEN) for s in SEGMENTS}
seg_load    = {s: 0   for s in SEGMENTS}
seg_pred    = {s: 0.0 for s in SEGMENTS}

def update_occupancy(trains):
    for s in SEGMENTS:
        seg_load[s] = 0
    for train in trains:
        e = train.current_edge()
        if e:
            s = canonical_edge(*e)
            seg_load[s] += 1
    for s in SEGMENTS:
        seg_history[s].append(seg_load[s])
        recent      = list(seg_history[s])[-10:]
        avg         = np.mean(recent) / MAX_ON_SEG
        curr        = seg_load[s]    / MAX_ON_SEG
        seg_pred[s] = float(np.clip(0.6*avg + 0.4*curr, 0.0, 1.0))


CONGESTION_PENALTY = 15.0

def build_cost_graph():
    cost = {}
    for u, v, d in G.edges(data=True):
        base    = d.get("weight", 3)
        s       = canonical_edge(u, v)
        penalty = CONGESTION_PENALTY * seg_pred[s] if seg_pred[s] > 0.40 else 0
        w = base + penalty
        cost.setdefault(u, {})[v] = w
        cost.setdefault(v, {})[u] = w
    return cost

def dijkstra(graph, src, dst):
    dist = {src: 0}
    prev = {}
    pq   = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, float("inf")):
            continue
        if u == dst:
            break
        for v, w in graph.get(u, {}).items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    path, cur = [], dst
    while cur != src:
        path.append(cur)
        cur = prev.get(cur)
        if cur is None:
            return None
    path.append(src)
    return list(reversed(path))

def mandatory_route(waypoints):
    cg   = build_cost_graph()
    full = []
    for i in range(len(waypoints)-1):
        seg = dijkstra(cg, waypoints[i], waypoints[i+1])
        if seg is None:
            return None
        full = full[:-1] + seg if full else seg
    return full


TRAIN_MANDATORY = {

    "T1": ["W", "C", "E"],    # Express:  west → centre → east
    "T2": ["E", "C", "W"],    # Regional: east → centre → west  ← head-on T1

    "T3": ["S", "C", "W"],    # Regional: south → centre → west
    "T6": ["C", "S", "E"],    # Freight:  centre → south → east ← head-on T3

    "T4": ["W", "S", "E"],    # Regional: west → south → east
    "T5": ["C", "W", "S"],    # Freight:  west → centre → south (northern then south)
}

# Priority: 1=Express (never rerouted, others yield)
#           2=Regional (rerouted only when needed)
#           3=Freight  (rerouted first — lowest priority)
TRAIN_PRIORITY = {
    "T1": 1,   # Express  — never rerouted
    "T2": 2,   # Regional
    "T3": 2,   # Regional
    "T4": 2,   # Regional
    "T5": 3,   # Freight
    "T6": 3,   # Freight
}

class Train2D:
    def __init__(self, tid, speed_range=(0.010, 0.020)):
        self.train_id        = tid
        self.mandatory       = TRAIN_MANDATORY[tid]
        self.path            = mandatory_route(self.mandatory) or self.mandatory[:]
        self.edge_index      = 0

        # Fixed stagger — each train starts at different point on first edge
        # so they are visually separated at frame 0 (no pile-up at stations)
        _stagger = {"T1":0.10,"T2":0.10,"T3":0.10,"T4":0.40,"T5":0.75,"T6":0.50}
        self.progress        = _stagger.get(tid, 0.05)
        self.base_speed      = random.uniform(*speed_range)
        self.speed           = self.base_speed
        self.color           = "#4CAF50"
        self.history         = deque(maxlen=50)
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.velocity        = 0.0
        self.collision_risk  = 0.0
        self.reroute_count   = 0
        self.last_reroute    = -9999
        self.reroute_flash   = 0
        # Priority level from TRAIN_PRIORITY — lower = higher priority
        self.priority        = TRAIN_PRIORITY.get(tid, 3)

        self.stopped_headon  = False
        self.headon_timer    = 0

    def current_edge(self):
        if self.edge_index >= len(self.path)-1:
            return None
        return (self.path[self.edge_index], self.path[self.edge_index+1])

    def apply_brake(self, intensity):
        self.is_braking      = True
        self.brake_intensity = intensity
        self.speed           = self.base_speed * (1 - intensity*0.8)

    def full_stop(self, frames=40):
        """Completely stop train for given number of frames (head-on scenario)."""
        self.stopped_headon  = True
        self.headon_timer    = frames
        self.speed           = 0.0
        self.brake_intensity = 1.0
        self.is_braking      = True

    def release_brake(self):
        self.is_braking      = False
        self.brake_intensity = 0.0
        self.speed           = self.base_speed

    def update(self):

        if self.stopped_headon:
            self.headon_timer -= 1
            if self.headon_timer <= 0:
                self.stopped_headon  = False
                self.brake_intensity = 0.0
                self.is_braking      = False
                self.speed           = self.base_speed
            return

        self.progress += self.speed
        if self.progress >= 1:
            self.progress   = 0
            self.edge_index = (self.edge_index+1) % max(1, len(self.path)-1)
        if self.reroute_flash > 0:
            self.reroute_flash -= 1

    def get_position(self):
        if self.edge_index >= len(self.path)-1:
            return pos[self.path[-1]]
        x1,y1 = pos[self.path[self.edge_index]]
        x2,y2 = pos[self.path[self.edge_index+1]]
        return (x1+(x2-x1)*self.progress, y1+(y2-y1)*self.progress)

    def _score_path(self, p):
        total = 0.0
        for i in range(len(p)-1):
            s = canonical_edge(p[i], p[i+1])
            total += seg_pred.get(s, 0.0)
        return total

    def choose_best_path(self, src, dst):
        """Enumerate all simple paths src→dst, return least-congested."""
        try:
            all_paths = list(nx.all_simple_paths(G, src, dst, cutoff=6))
        except Exception:
            all_paths = []
        visited          = set(self.path[:self.edge_index+1])
        unvisited_majors = set(self.mandatory) - visited - {src, dst}
        valid = [p for p in all_paths
                 if not set(p[1:-1]).intersection(unvisited_majors)]
        if not valid:
            return dijkstra(build_cost_graph(), src, dst)
        return min(valid, key=self._score_path)

    def plan_from_major(self, frame):
        """
        At a major station: pick least-congested path to next mandatory stop.
        Only reroutes when current path is genuinely congested AND a better
        alternative exists (>30% improvement).
        """
        if frame - self.last_reroute < 30:
            return False
        current_node = self.path[self.edge_index]
        if current_node not in MAJOR_STATIONS:
            return False
        visited_majors = {n for n in self.path[:self.edge_index+1] if n in MAJOR_STATIONS}
        remaining = [m for m in self.mandatory if m not in visited_majors]
        if not remaining:
            return False

        next_major = remaining[0]

        try:
            nm_idx      = self.path[self.edge_index:].index(next_major)
            current_seg = self.path[self.edge_index: self.edge_index+nm_idx+1]
        except ValueError:
            current_seg = []
        current_score = self._score_path(current_seg)

        path_len = max(len(current_seg)-1, 1)
        avg_load = current_score / path_len
        if avg_load < PATH_REROUTE_THRESH:
            return False

        best_path  = self.choose_best_path(current_node, next_major)
        if best_path is None:
            return False
        best_score = self._score_path(best_path)
        if best_score >= current_score * 0.90:
            return False   # not worth switching (only skip if < 10% better)

        after = remaining[1:]
        tail  = mandatory_route([next_major]+after) if after else []
        new_suffix = best_path + (tail[1:] if tail else [])
        if new_suffix == self.path[self.edge_index:]:
            return False

        self.path          = self.path[:self.edge_index] + new_suffix
        self.progress      = 0
        self.reroute_count += 1
        self.last_reroute  = frame
        self.reroute_flash = 25
        return True


HEAD_ON_DIST   = 1.20
HEAD_ON_FRAMES = 60

def detect_and_handle_headon(trains):
    handled = set()
    for i, t1 in enumerate(trains):
        if t1.stopped_headon:
            continue
        e1 = t1.current_edge()
        if e1 is None:
            continue
        for t2 in trains[i+1:]:
            if t2.stopped_headon:
                continue
            e2 = t2.current_edge()
            if e2 is None:
                continue
            # Head-on: same physical edge, opposite traversal direction
            if e2 == (e1[1], e1[0]):
                d = np.linalg.norm(
                    np.array(t1.get_position()) - np.array(t2.get_position())
                )
                if d < HEAD_ON_DIST and (t1.train_id, t2.train_id) not in handled:
                    t1.full_stop(HEAD_ON_FRAMES)
                    t2.full_stop(HEAD_ON_FRAMES)
                    handled.add((t1.train_id, t2.train_id))
                    handled.add((t2.train_id, t1.train_id))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

risk_model = model.to(device)
risk_model.eval()

try:
    with open("scaler.pkl","rb") as f:
        _scaler = pickle.load(f)
except FileNotFoundError:
    _scaler = None

try:
    with open("model_config.json") as f:
        _cfg = json.load(f)
    _THRESHOLD = _cfg.get("optimal_threshold", 0.5)
except FileNotFoundError:
    _THRESHOLD = 0.5

_SEG_PROPS = {
    0: {"gradient":-0.005,"curvature":0.005,"speed_limit":34.6,"friction":0.88},
    1: {"gradient":-0.014,"curvature":0.001,"speed_limit":21.2,"friction":0.96},
    2: {"gradient": 0.004,"curvature":0.004,"speed_limit":20.4,"friction":0.99},
    3: {"gradient": 0.013,"curvature":0.001,"speed_limit":23.6,"friction":0.76},
    4: {"gradient":-0.008,"curvature":0.003,"speed_limit":28.6,"friction":0.79},
    5: {"gradient": 0.004,"curvature":0.001,"speed_limit":25.8,"friction":0.81},
    6: {"gradient":-0.002,"curvature":0.004,"speed_limit":24.0,"friction":0.85},
    7: {"gradient": 0.004,"curvature":0.000,"speed_limit":32.2,"friction":0.75},
}

CRITICAL_DIST = 0.35
WARNING_DIST  = 0.70
SAFE_DIST     = 1.50
SEQ_LEN       = 10

def compute_risk(train, others, frame=0):
    min_headway = 10000.0
    ce = train.current_edge()
    if ce is None:
        return 0.0

    for other in others:
        if other.train_id == train.train_id:
            continue
        oe = other.current_edge()
        # Same edge either direction
        if oe and (oe == ce or oe == (ce[1], ce[0])):
            d = np.linalg.norm(
                np.array(train.get_position()) - np.array(other.get_position())
            )
            min_headway = min(min_headway, d*100)

    velocity     = train.speed * 100.0
    acceleration = (train.speed - train.base_speed) * 100.0
    headway      = float(np.clip(min_headway, 0, 500))

    seg_idx  = train.edge_index % len(_SEG_PROPS)
    props    = _SEG_PROPS[seg_idx]

    if   min_headway < 100:  signal_state = 2.0
    elif min_headway < 250:  signal_state = 1.0
    else:                    signal_state = 0.0

    raw = np.array([
        velocity, acceleration, headway,
        props["gradient"], props["curvature"], props["speed_limit"],
        signal_state, props["friction"],
    ], dtype=np.float32)

    if _scaler is not None:
        feat = _scaler.transform(raw.reshape(1,-1))[0].astype(np.float32)
    else:
        feat = raw

    train.history.append(feat)

    if len(train.history) >= SEQ_LEN:
        seq = np.array(list(train.history)[-SEQ_LEN:])
        t   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            prob = torch.sigmoid(risk_model(t)).item()
    else:
        prob = 1.0 if min_headway < CRITICAL_DIST*100 else (
               0.5 if min_headway < WARNING_DIST*100  else 0.1)

    train.velocity       = velocity
    train.collision_risk = prob

    dist_m = min_headway
    if   dist_m < CRITICAL_DIST*100:
        train.apply_brake(min(1.0,(CRITICAL_DIST*100-dist_m)/(CRITICAL_DIST*100)))
    elif dist_m < WARNING_DIST*100:
        train.apply_brake(0.5)
    elif dist_m < SAFE_DIST*100:
        train.apply_brake(0.2)
    elif not train.stopped_headon:
        train.release_brake()

    return prob


def reroute_trains_sequentially(trains, frame):
    """
    Exactly ONE train reroutes per frame in priority order.
    Freight (3) reroutes before Regional (2). Express (1) never reroutes.
    After the chosen train commits, occupancy is updated immediately so
    the next frame's candidate sees the reserved path and picks differently.
    """
    candidates = []
    for train in trains:
        if train.stopped_headon:                              continue
        if train.priority == 1:                              continue
        if train.path[train.edge_index] not in MAJOR_STATIONS: continue
        if train.progress >= 0.20:                           continue
        if frame - train.last_reroute < 30:                  continue

        # Find next unvisited mandatory major station
        visited_majors = {n for n in train.path[:train.edge_index+1]
                          if n in MAJOR_STATIONS}
        remaining = [m for m in train.mandatory if m not in visited_majors]
        if not remaining:
            continue
        next_major = remaining[0]
        try:
            nm_idx      = train.path[train.edge_index:].index(next_major)
            current_seg = train.path[train.edge_index: train.edge_index+nm_idx+1]
        except ValueError:
            continue
        avg_load = train._score_path(current_seg) / max(len(current_seg)-1, 1)
        if avg_load < PATH_REROUTE_THRESH:
            continue
        candidates.append((train.priority, id(train), train, remaining, next_major, current_seg))

    if not candidates:
        return

    # Highest priority number (freight=3) goes first — sort descending by priority
    candidates.sort(key=lambda x: -x[0])
    _, _, chosen, remaining, next_major, current_seg = candidates[0]

    # Find best alternative path
    best_path = chosen.choose_best_path(chosen.path[chosen.edge_index], next_major)
    if best_path is None:
        return

    current_score = chosen._score_path(current_seg)
    best_score    = chosen._score_path(best_path)

    # Only switch if meaningfully better (at least 10% improvement)
    if best_score >= current_score * 0.90:
        return

    after      = remaining[1:]
    tail       = mandatory_route([next_major] + after) if after else []
    new_suffix = best_path + (tail[1:] if tail else [])
    if new_suffix == chosen.path[chosen.edge_index:]:
        return


    chosen.path          = chosen.path[:chosen.edge_index] + new_suffix
    chosen.progress      = 0
    chosen.reroute_count += 1
    chosen.last_reroute  = frame
    chosen.reroute_flash = 25
    update_occupancy(trains)


speed_ranges = {
    "T1":(0.013,0.018),  # Express — fastest
    "T2":(0.011,0.016),  # Regional
    "T3":(0.011,0.016),  # Regional
    "T4":(0.010,0.015),  # Regional
    "T5":(0.009,0.014),  # Freight — slowest
    "T6":(0.009,0.014),  # Freight — slowest
}
trains = [Train2D(tid, speed_ranges[tid]) for tid in TRAIN_MANDATORY]

print(f"✅ {len(trains)} trains | {len(G.nodes())} stations | {len(G.edges())} segments | device={device}")

#DASHBOARD

plt.style.use('dark_background')
fig = plt.figure(figsize=(28, 16), dpi=90)
gs  = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.38)

ax_net     = fig.add_subplot(gs[:2, :2])
ax_risk    = fig.add_subplot(gs[0,  2])
ax_speed   = fig.add_subplot(gs[1,  2])
ax_occ     = fig.add_subplot(gs[0,  3])
ax_head    = fig.add_subplot(gs[2,  0])
ax_brake   = fig.add_subplot(gs[2,  1])
ax_reroute = fig.add_subplot(gs[2,  2])
ax_stats   = fig.add_subplot(gs[2,  3])

# ── Network base ──────────────────────────────────────────────
nx.draw_networkx_edges(G, pos=pos, ax=ax_net,
                       edge_color='#37474F', width=7, alpha=0.6)

# Major stations — gold squares
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MAJOR_STATIONS),
                       node_color='#FFD700', node_size=2200,
                       node_shape='s', alpha=0.95)
# Minor waypoints — grey circles
nx.draw_networkx_nodes(G, pos=pos, ax=ax_net,
                       nodelist=list(MINOR_STATIONS),
                       node_color='#546E7A', node_size=600,
                       node_shape='o', alpha=0.80)
nx.draw_networkx_labels(G, pos=pos, ax=ax_net,
                        font_color='white', font_weight='bold', font_size=12)

# ── Signal light markers on major stations ────────────────────
SIGNAL_COLORS = {0: '#00E676', 1: '#FFD740', 2: '#FF1744'}  # green, yellow, red
signal_dots = {}
for station in MAJOR_STATIONS:
    sx, sy = pos[station]
    dot = ax_net.scatter(sx, sy+0.55, s=220, zorder=10,
                         color=SIGNAL_COLORS[0],
                         edgecolors='white', linewidths=1.5)
    signal_dots[station] = dot

# Signal legend
sig_patches = [
    mpatches.Patch(color='#00E676', label='Signal: Green (clear)'),
    mpatches.Patch(color='#FFD740', label='Signal: Yellow (caution)'),
    mpatches.Patch(color='#FF1744', label='Signal: Red (danger)'),
    mpatches.Patch(color='#FFD700', label='Major Station'),
    mpatches.Patch(color='#546E7A', label='Minor Waypoint'),
]
ax_net.legend(handles=sig_patches, loc='upper right', fontsize=8,
              framealpha=0.7)

# ── Track load overlays — coloured lines per segment ─────────
load_cmap  = plt.cm.RdYlGn_r
edge_lines = {}
for (u,v) in SEGMENTS:
    x1,y1 = pos[u]; x2,y2 = pos[v]
    ln, = ax_net.plot([x1,x2],[y1,y2], lw=9, alpha=0.0, zorder=2,
                      solid_capstyle='round')
    edge_lines[(u,v)] = ln

# ── Train scatter + indicators ────────────────────────────────
colors_t     = ['#00E5FF','#FF6D00','#69FF47','#FF4DFF','#FFD700','#FF1744']
train_scat   = ax_net.scatter([], [], s=520, zorder=6,
                               edgecolors='white', linewidths=2.5)

# Brake indicator — red pulsing ring
brake_circles = [Circle((0,0), 0.38, fill=False, edgecolor='#FF1744',
                          linewidth=4, visible=False, zorder=8) for _ in trains]
# Reroute flash — cyan dashed ring
reroute_circles = [Circle((0,0), 0.52, fill=False, edgecolor='#00E5FF',
                            linewidth=3, linestyle='--', visible=False, zorder=8)
                   for _ in trains]
# Head-on stop — white solid ring (larger)
headon_circles = [Circle((0,0), 0.62, fill=False, edgecolor='#FFFFFF',
                           linewidth=4, visible=False, zorder=8) for _ in trains]

for c in brake_circles + reroute_circles + headon_circles:
    ax_net.add_patch(c)

tlabels = [ax_net.text(0,0,'', fontsize=9, color='white', ha='center',
                        va='bottom', fontweight='bold', zorder=9)
           for _ in trains]

ax_net.set_title("🚂  Railway Network — Signals, Braking, Rerouting",
                  fontsize=14, fontweight='bold', pad=12)
ax_net.set_xlim(-1.5, 14); ax_net.set_ylim(-1, 10.5); ax_net.axis('off')

# ── Risk / Speed lines ────────────────────────────────────────
risk_lines  = [ax_risk.plot([],[],  color=c, lw=2, label=t.train_id)[0]
               for t,c in zip(trains,colors_t)]
speed_lines = [ax_speed.plot([],[], color=c, lw=2, label=t.train_id)[0]
               for t,c in zip(trains,colors_t)]
for ax, ttl, yl in [(ax_risk,"⚠️  Collision Risk","Risk (0–1)"),
                     (ax_speed,"🚄  Velocities","Speed")]:
    ax.set_title(ttl, fontsize=11, fontweight='bold')
    ax.set_ylabel(yl, fontsize=9)
    ax.legend(loc='upper left', fontsize=7, ncol=2)
    ax.grid(True, alpha=0.25)
ax_risk.set_ylim(0, 1)

# ── Occupancy heatmap ─────────────────────────────────────────
OCC_WIN  = 40
occ_data = np.zeros((len(SEGMENTS), OCC_WIN))
occ_im   = ax_occ.imshow(occ_data, aspect='auto', cmap='RdYlGn_r',
                          vmin=0, vmax=1, interpolation='nearest', origin='upper')
ax_occ.set_yticks(range(len(SEGMENTS)))
ax_occ.set_yticklabels([f"{u}–{v}" for u,v in SEGMENTS], fontsize=7)
ax_occ.set_xticks([0, OCC_WIN//2, OCC_WIN-1])
ax_occ.set_xticklabels([f"–{OCC_WIN}","",  "now"], fontsize=8)
ax_occ.set_title("📡  Track Occupancy\n(green=free | red=busy)",
                  fontsize=10, fontweight='bold')
plt.colorbar(occ_im, ax=ax_occ, fraction=0.05, pad=0.03).set_label("Congestion", fontsize=8)

# ── Headway ───────────────────────────────────────────────────
hw_line, = ax_head.plot([],[], 'cyan', lw=2.5, label='Min Headway')
ax_head.axhline(CRITICAL_DIST, color='red',    ls='--', lw=2, label='Critical/Stop')
ax_head.axhline(WARNING_DIST,  color='orange', ls='--', lw=2, label='Warning')
ax_head.set_title("📏  Minimum Headway (grid units)", fontsize=11, fontweight='bold')
ax_head.set_ylabel("Distance", fontsize=9); ax_head.set_xlabel("Frame", fontsize=9)
ax_head.legend(fontsize=8); ax_head.grid(True, alpha=0.25)
ax_head.set_ylim(0, 8)

# ── Brake bars ────────────────────────────────────────────────
brake_bars = ax_brake.barh(range(len(trains)), [0]*len(trains),
                            color='#FF5252', alpha=0.82, height=0.6)
ax_brake.set_yticks(range(len(trains)))
ax_brake.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_brake.set_xlim(0, 1.1)
ax_brake.set_title("🛑  Brake Intensity", fontsize=11, fontweight='bold')
ax_brake.set_xlabel("0 = None  →  1 = Full Stop", fontsize=9)
ax_brake.grid(True, alpha=0.25, axis='x')
brake_txt = [ax_brake.text(0.02, i, '0.00', va='center', fontsize=8, color='white')
             for i in range(len(trains))]

# ── Reroute bars ──────────────────────────────────────────────
reroute_bars = ax_reroute.barh(range(len(trains)), [0]*len(trains),
                                color='#29B6F6', alpha=0.85, height=0.6)
ax_reroute.set_yticks(range(len(trains)))
ax_reroute.set_yticklabels([t.train_id for t in trains], fontsize=9)
ax_reroute.set_xlim(0, 12)
ax_reroute.set_title("🔀  Reroute Events", fontsize=11, fontweight='bold')
ax_reroute.set_xlabel("Times rerouted", fontsize=9)
ax_reroute.grid(True, alpha=0.25, axis='x')
reroute_txt = [ax_reroute.text(0.15, i, '0', va='center', fontsize=9,
                                color='white', fontweight='bold')
               for i in range(len(trains))]

ax_stats.axis('off')

risk_hist  = {t.train_id: [] for t in trains}
speed_hist = {t.train_id: [] for t in trains}
hw_hist, ts = [], []

print("\nDashboard ready. Building animation …")

# ══════════════════════════════════════════════════════════════
# 11. ANIMATION
# ══════════════════════════════════════════════════════════════
def update(frame):
    # ─ 1. occupancy + signals ─────────────────────────────────
    update_occupancy(trains)
    update_signals(trains)

    # ─ 2. heatmap ─────────────────────────────────────────────
    occ_data[:, :-1] = occ_data[:, 1:]
    for si, seg in enumerate(SEGMENTS):
        occ_data[si, -1] = seg_pred[seg]
    occ_im.set_data(occ_data)

    # ─ 3. track load colours ──────────────────────────────────
    for seg, ln in edge_lines.items():
        load = seg_pred[seg]
        ln.set_color(load_cmap(load))
        ln.set_alpha(0.15 + 0.75*load)

    # ─ 4. update signal dots on map ───────────────────────────
    for station, dot in signal_dots.items():
        dot.set_color(SIGNAL_COLORS[station_signal[station]])

    # ─ 5. head-on detection ───────────────────────────────────
    detect_and_handle_headon(trains)

    # ─ 6. update trains ───────────────────────────────────────
    xs, ys, cols = [], [], []
    risks, speeds = [], []
    braking_count = 0

    for i, train in enumerate(trains):
        train.update()
        risk = compute_risk(train, trains, frame)

        # Rerouting handled sequentially outside this loop (see below)

        risk_hist[train.train_id].append(risk)
        speed_hist[train.train_id].append(train.velocity)
        risks.append(risk); speeds.append(train.velocity)

        x, y = train.get_position()

        # Head-on stop ring — white solid, largest
        if train.stopped_headon:
            headon_circles[i].set_center((x, y))
            headon_circles[i].set_visible(True)
            brake_circles[i].set_visible(False)
        else:
            headon_circles[i].set_visible(False)

        # Brake ring — red
        if train.is_braking and not train.stopped_headon:
            braking_count += 1
            brake_circles[i].set_center((x, y))
            brake_circles[i].set_visible(True)
        else:
            if not train.stopped_headon:
                brake_circles[i].set_visible(False)

        # Reroute flash ring — cyan dashed
        if train.reroute_flash > 0:
            reroute_circles[i].set_center((x, y))
            reroute_circles[i].set_visible(True)
        else:
            reroute_circles[i].set_visible(False)

        # Train colour by risk level
        if   train.stopped_headon: train.color = "#FFFFFF"   # white = stopped
        elif risk > _THRESHOLD:    train.color = "#FF1744"   # red
        elif risk > 0.40:          train.color = "#FF9800"   # orange
        elif risk > 0.20:          train.color = "#FFEB3B"   # yellow
        else:                      train.color = "#4CAF50"   # green

        xs.append(x); ys.append(y); cols.append(train.color)
        tlabels[i].set_position((x, y+0.48))
        prio_sym = "★" if train.priority == 1 else ("◆" if train.priority == 2 else "●")
        tlabels[i].set_text(f"{train.train_id}{prio_sym}")

        if train.is_braking and not train.stopped_headon:
            braking_count += 1

    train_scat.set_offsets(np.c_[xs, ys])
    train_scat.set_color(cols)

    # ─ 6b. sequential rerouting — one train per frame, priority order ──
    reroute_trains_sequentially(trains, frame)

    # ─ 7. headway ─────────────────────────────────────────────
    min_hw = min(
        np.linalg.norm(np.array(t1.get_position()) - np.array(t2.get_position()))
        for ii, t1 in enumerate(trains) for t2 in trains[ii+1:]
    )
    hw_hist.append(min_hw); ts.append(frame)
    hw_line.set_data(ts, hw_hist)
    ax_head.set_xlim(0, max(50, frame+1))

    # ─ 8. risk / speed plots ──────────────────────────────────
    for i, train in enumerate(trains):
        h = risk_hist[train.train_id]
        risk_lines[i].set_data(range(len(h)), h)
        s = speed_hist[train.train_id]
        speed_lines[i].set_data(range(len(s)), s)
    ax_risk.set_xlim(0,  max(50, frame+1))
    ax_speed.set_xlim(0, max(50, frame+1))
    if speeds:
        ax_speed.set_ylim(0, max(speeds+[1])*1.15)

    # ─ 9. brake bars ──────────────────────────────────────────
    for j, (bar, train) in enumerate(zip(brake_bars, trains)):
        bar.set_width(train.brake_intensity)
        brake_txt[j].set_x(train.brake_intensity + 0.01)
        brake_txt[j].set_text(f"{train.brake_intensity:.2f}")

    # ─ 10. reroute bars ───────────────────────────────────────
    max_r = max((t.reroute_count for t in trains), default=0)
    ax_reroute.set_xlim(0, max(12, max_r+2))
    for j, (bar, train) in enumerate(zip(reroute_bars, trains)):
        bar.set_width(train.reroute_count)
        reroute_txt[j].set_x(train.reroute_count + 0.12)
        reroute_txt[j].set_text(str(train.reroute_count))

    # ─ 11. stats panel ────────────────────────────────────────
    ax_stats.clear(); ax_stats.axis('off')
    stopped  = sum(1 for t in trains if t.stopped_headon)
    total_r  = sum(t.reroute_count for t in trains)
    sig_str  = " ".join(
        f"{s}:{'🔴' if station_signal[s]==2 else '🟡' if station_signal[s]==1 else '🟢'}"
        for s in sorted(MAJOR_STATIONS)
    )
    status   = ("🔴 HEAD-ON STOP" if stopped > 0 else
                "🔴 ALERT!"       if min_hw < CRITICAL_DIST else
                "🟡 Caution"      if min_hw < WARNING_DIST  else
                "🟢 Normal")
    ax_stats.text(0.04, 0.5,
        f"  ⏱  Frame:       {frame}\n\n"
        f"  🚂 Trains:      {len(trains)}\n"
        f"  🛑 Braking:     {braking_count}\n"
        f"  ⛔ Head-on Stop:{stopped}\n"
        f"  🔀 Reroutes:    {total_r}\n\n"
        f"  📏 Min Headway: {min_hw:.2f}\n"
        f"  📊 Avg Risk:    {np.mean(risks):.3f}\n"
        f"  ⚠️  Max Risk:    {max(risks):.3f}\n\n"
        f"  Signals:\n  {sig_str}\n\n"
        f"  {status}",
        fontsize=9, va='center', family='monospace', color='white',
        transform=ax_stats.transAxes)

    return ([train_scat, occ_im, hw_line]
            + list(edge_lines.values())
            + list(signal_dots.values())
            + brake_circles + reroute_circles + headon_circles
            + risk_lines + speed_lines + tlabels)

# ══════════════════════════════════════════════════════════════
# 12. RUN
# ══════════════════════════════════════════════════════════════
ani = FuncAnimation(fig, update, frames=1200, interval=50, blit=False)
plt.close()


HTML(ani.to_jshtml())

✅ 6 trains | 11 stations | 14 segments | device=cuda

Dashboard ready. Building animation …


In [ ]:
ani.save('railway_demo.mp4', writer='ffmpeg', fps=20, dpi=90,
         extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
                     '-preset', 'medium', '-crf', '23'])